# Participant name matching

This notebook compares participant names extracted from `outputs/**/*.json` against the per-conference truth files in `analysis_v1/data/*/*_outcome.json`, then writes the matched and unmatched participant CSVs into `analysis_v1/`. Name corrections are optional and only applied if a mapping file exists.

In [1]:
import json
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

def resolve_base():
    cwd = Path.cwd().resolve()

    if cwd.name == "analysis_v1":
        return cwd.parent

    if (cwd / "analysis_v1").exists() and (cwd / "outputs").exists():
        return cwd

    if (cwd.parent / "analysis_v1").exists() and (cwd.parent / "outputs").exists():
        return cwd.parent

    raise FileNotFoundError(
        f"Could not locate the repo root from current working directory: {cwd}. "
        "Run the notebook from the repo root or from analysis_v1/ under it."
    )

BASE = resolve_base()
REPO_ROOT = BASE
OUTPUTS_ROOT = REPO_ROOT / "outputs"
TRUTH_ROOT = REPO_ROOT / "analysis_v1" / "data"
NAME_MAPPING_CANDIDATES = [
    REPO_ROOT / "analysis_v1" / "name_corrections_mapping.json", 
    REPO_ROOT / "name_corrections_mapping.json",
]
MANUAL_ALIAS_CANDIDATES = [
    REPO_ROOT / "analysis_v1" / "participant_alias_mapping.csv",
    REPO_ROOT / "participant_alias_mapping.csv",
]

if not OUTPUTS_ROOT.exists():
    raise FileNotFoundError(f"Outputs folder not found: {OUTPUTS_ROOT}")

if not TRUTH_ROOT.exists():
    raise FileNotFoundError(f"Truth data folder not found: {TRUTH_ROOT}")

def load_json(path: Path):
    try:
        with path.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    except json.JSONDecodeError as exc:
        print(f"Skipping invalid JSON file: {path} ({exc})")
        return None

def strip_unicode(value):
    text = unicodedata.normalize("NFKD", value)
    return text.encode("ascii", "ignore").decode("ascii")

def normalize_name(value):
    if not isinstance(value, str):
        return ""
    text = strip_unicode(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

NAME_PREFIXES = {
    "dr",
    "prof",
    "professor",
    "mr",
    "mrs",
    "ms",
    "miss",
    "mx",
}

NAME_SUFFIXES = {
    "phd",
    "md",
    "m d",
    "dds",
    "dmd",
    "do",
    "msc",
    "ma",
    "ba",
    "bs",
    "mba",
    "jd",
    "esq",
}

AFFILIATION_WORDS = {
    "university",
    "univ",
    "college",
    "institute",
    "school",
    "department",
    "dept",
    "center",
    "centre",
    "laboratory",
    "lab",
    "labs",
    "hospital",
    "company",
    "industry",
    "research",
    "medicine",
    "med",
    "engineering",
}

ROLE_WORDS = {
    "she",
    "her",
    "he",
    "him",
    "they",
    "them",
    "their",
    "his",
    "hers",
    "speaker",
    "moderator",
    "participant",
}

def clean_name_for_matching(value):
    if not isinstance(value, str):
        return ""

    text = strip_unicode(value).lower().strip()
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\b(?:dr|prof|professor|mr|mrs|ms|miss|mx)\.?\s+", "", text)
    text = re.sub(r"[|;/]+", " ", text)
    text = re.sub(r"\s*,\s*", ",", text)
    text = re.sub(r"\s+", " ", text).strip()

    parts = [part.strip() for part in text.split(",") if part.strip()]
    if parts:
        text = parts[0]
    else:
        text = text.replace(",", " ")

    tokens = text.split()
    if not tokens:
        return ""

    while tokens and tokens[0].rstrip(".") in NAME_PREFIXES:
        tokens.pop(0)

    while tokens and tokens[-1].rstrip(".") in NAME_SUFFIXES.union(ROLE_WORDS):
        tokens.pop()

    cutoff = len(tokens)
    for index, token in enumerate(tokens):
        if token.rstrip(".") in AFFILIATION_WORDS:
            cutoff = index
            break

    if cutoff > 0:
        tokens = tokens[:cutoff]

    while tokens and tokens[-1].rstrip(".") in NAME_SUFFIXES.union(ROLE_WORDS):
        tokens.pop()

    return normalize_name(" ".join(tokens))

def build_corrections(mapping_data):
    corrections = {}

    def add(alias, canonical):
        alias_norm = clean_name_for_matching(alias)
        canonical_norm = clean_name_for_matching(canonical)
        if alias_norm and canonical_norm:
            corrections[alias_norm] = canonical_norm

    if isinstance(mapping_data, dict):
        for key, value in mapping_data.items():
            if isinstance(value, str):
                add(key, value)
            elif isinstance(value, list):
                for alias in value:
                    add(alias, key)
            elif isinstance(value, dict):
                canonical = (
                    value.get("corrected_name")
                    or value.get("canonical_name")
                    or value.get("canonical")
                    or value.get("name")
                    or key
                )
                aliases = (
                    value.get("aliases")
                    or value.get("variants")
                    or value.get("alternate_names")
                    or value.get("original_names")
                    or value.get("from")
                    or value.get("alias")
                )
                if isinstance(aliases, list):
                    for alias in aliases:
                        add(alias, canonical)
                else:
                    add(key, canonical)
    elif isinstance(mapping_data, list):
        for item in mapping_data:
            if not isinstance(item, dict):
                continue
            alias = item.get("alias") or item.get("from") or item.get("wrong") or item.get("name")
            canonical = item.get("corrected_name") or item.get("correct") or item.get("to") or item.get("canonical_name")
            add(alias, canonical)

    return corrections

def load_manual_alias_mapping():
    alias_path = next((path for path in MANUAL_ALIAS_CANDIDATES if path.exists()), None)
    if alias_path is None:
        return {}, None

    alias_df = pd.read_csv(alias_path)
    if alias_df.empty:
        return {}, alias_path

    alias_columns = {column.lower(): column for column in alias_df.columns}
    if "alias_name" not in alias_columns or "canonical_name" not in alias_columns:
        raise ValueError(
            f"Manual alias file {alias_path} must contain alias_name and canonical_name columns."
        )

    alias_name_column = alias_columns["alias_name"]
    canonical_name_column = alias_columns["canonical_name"]
    manual_aliases = {}
    for _, row in alias_df.iterrows():
        alias_name = row.get(alias_name_column)
        canonical_name = row.get(canonical_name_column)
        if pd.isna(alias_name) or pd.isna(canonical_name):
            continue
        alias_clean = clean_name_for_matching(str(alias_name))
        canonical_clean = clean_name_for_matching(str(canonical_name))
        if alias_clean and canonical_clean:
            manual_aliases[alias_clean] = canonical_clean

    return manual_aliases, alias_path

mapping_path = next((path for path in NAME_MAPPING_CANDIDATES if path.exists()), None)
if mapping_path is None:
    NAME_CORRECTIONS = {}
else:
    mapping_data = load_json(mapping_path)
    NAME_CORRECTIONS = build_corrections(mapping_data) if mapping_data is not None else {}

MANUAL_ALIAS_CORRECTIONS, MANUAL_ALIAS_PATH = load_manual_alias_mapping()

def canonicalize_name(value):
    normalized = clean_name_for_matching(value)
    if not normalized:
        return ""
    normalized = NAME_CORRECTIONS.get(normalized, normalized)
    normalized = MANUAL_ALIAS_CORRECTIONS.get(normalized, normalized)
    return normalized

In [2]:
TARGET_NAME_FIELDS = {
    "name",
    "participant",
    "participants",
    "participant_name",
    "full_name",
    "speaker",
    "speakers",
    "member",
    "members",
    "founder",
    "founders",
    "founder_name",
    "person",
    "people",
}

def collect_output_name_rows(data, conference, source_file):
    rows = []

    chunk_summary = data.get("chunk_summary") or {}
    speaking_time_seconds = chunk_summary.get("speaking_time_seconds") or {}
    if isinstance(speaking_time_seconds, dict):
        for name in speaking_time_seconds.keys():
            rows.append((conference, name, "chunk_summary.speaking_time_seconds", source_file))

    utterance_annotations = data.get("utterance_annotations") or []
    if isinstance(utterance_annotations, list):
        for index, utterance in enumerate(utterance_annotations):
            if isinstance(utterance, dict):
                speaker = utterance.get("speaker")
                if isinstance(speaker, str):
                    rows.append((conference, speaker, f"utterance_annotations[{index}].speaker", source_file))

    session_state = data.get("session_state") or {}
    speakers_identified = session_state.get("speakers_identified") or []
    if isinstance(speakers_identified, list):
        for speaker in speakers_identified:
            if isinstance(speaker, str):
                rows.append((conference, speaker, "session_state.speakers_identified", source_file))

    return rows

def extract_truth_name_rows(obj, conference, source_path="root"):
    rows = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            key_name = str(key).strip().lower()
            if key_name in TARGET_NAME_FIELDS:
                rows.extend(extract_names_from_value(value, conference, f"{source_path}.{key}"))
            rows.extend(extract_truth_name_rows(value, conference, f"{source_path}.{key}"))
    elif isinstance(obj, list):
        for index, item in enumerate(obj):
            rows.extend(extract_truth_name_rows(item, conference, f"{source_path}[{index}]"))

    return rows

def extract_names_from_value(value, conference, source_path):
    rows = []

    if isinstance(value, str):
        rows.append((conference, value, source_path))
    elif isinstance(value, list):
        for index, item in enumerate(value):
            if isinstance(item, str):
                rows.append((conference, item, source_path))
            elif isinstance(item, (dict, list)):
                rows.extend(extract_truth_name_rows(item, conference, f"{source_path}[{index}]"))
    elif isinstance(value, dict):
        rows.extend(extract_truth_name_rows(value, conference, source_path))

    return rows

def register_occurrence(store, conference, raw_name, source_label, source_file):
    canonical_name = canonicalize_name(raw_name)
    if not canonical_name:
        return

    key = (conference, canonical_name)
    entry = store[key]
    entry["raw_names"].add(raw_name.strip())
    entry["source_labels"].add(source_label)
    entry["source_files"].add(str(source_file.relative_to(REPO_ROOT)))
    entry["occurrence_count"] += 1

def make_empty_entry():
    return {
        "raw_names": set(),
        "source_labels": set(),
        "source_files": set(),
        "occurrence_count": 0,
    }

def store_to_dataframe(store, prefix):
    columns = [
        "conference",
        "normalized_name",
        f"{prefix}_raw_names",
        f"{prefix}_sources",
        f"{prefix}_source_files",
        f"{prefix}_occurrence_count",
        f"{prefix}_unique_file_count",
    ]
    records = []
    for (conference, normalized_name), payload in sorted(store.items()):
        records.append({
            "conference": conference,
            "normalized_name": normalized_name,
            f"{prefix}_raw_names": " | ".join(sorted(payload["raw_names"])),
            f"{prefix}_sources": " | ".join(sorted(payload["source_labels"])),
            f"{prefix}_source_files": " | ".join(sorted(payload["source_files"])),
            f"{prefix}_occurrence_count": payload["occurrence_count"],
            f"{prefix}_unique_file_count": len(payload["source_files"]),
        })

    if not records:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(records, columns=columns)

output_store = defaultdict(make_empty_entry)
truth_store = defaultdict(make_empty_entry)

output_files = sorted(OUTPUTS_ROOT.glob("**/*.json"))
truth_files = sorted(TRUTH_ROOT.glob("*/*_outcome.json"))
skipped_output_files = []
skipped_truth_files = []

for output_file in output_files:
    relative_parts = output_file.relative_to(OUTPUTS_ROOT).parts
    if not relative_parts:
        continue
    conference = relative_parts[0]
    data = load_json(output_file)
    if data is None:
        skipped_output_files.append(str(output_file))
        continue
    for _, raw_name, source_label, source_path in collect_output_name_rows(data, conference, output_file):
        register_occurrence(output_store, conference, raw_name, source_label, source_path)

for truth_file in truth_files:
    conference = truth_file.parent.name
    data = load_json(truth_file)
    if data is None:
        skipped_truth_files.append(str(truth_file))
        continue
    for _, raw_name, source_label in extract_truth_name_rows(data, conference):
        register_occurrence(truth_store, conference, raw_name, source_label, truth_file)

output_df = store_to_dataframe(output_store, "output")
truth_df = store_to_dataframe(truth_store, "truth")

output_df = output_df.sort_values(["conference", "normalized_name"]).reset_index(drop=True) if not output_df.empty else output_df
truth_df = truth_df.sort_values(["conference", "normalized_name"]).reset_index(drop=True) if not truth_df.empty else truth_df

if skipped_output_files:
    print(f"Skipped {len(skipped_output_files)} invalid output JSON files.")
if skipped_truth_files:
    print(f"Skipped {len(skipped_truth_files)} invalid truth JSON files.")

output_df.head(), truth_df.head()

Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2020NES/output_2020_11_06_NES_S6/6_CO2_Reduction_Zoom_Meeting_2020_11_06_08_45_55/ATTN_6_CO2_Reduction_Zoom_Meeting_2020_11_06_08_45_55.json (Expecting value: line 1 column 1 (char 0))


Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021MND/output_2021_04_22_MND_S8/ATTN_Zoom_Meeting_Room_6_2021_04_22_13_00_55_chunk2.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021MND/output_2021_04_22_MND_S8/ATTN_Zoom_Meeting_Room_6_2021_04_22_13_18_39_chunk1.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021MND/output_2021_04_23_MND_S11/ATTN_bot2_Zoom_Meeting_2021_04_23_13_13_56_chunk4.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021MND/output_2021_04_23_MND_S13/ATTN_botB1_2021_04_23_13_14_18_chunk3.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021MND/output_2021_04_23_MND_S8/ATTN_Zoom_Meeti

Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021NES/output_2021_11_05_NES_S3/ATTN_B3_2021_11_05_11_01_14_trimend_chunk6.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021NES/output_2021_11_05_NES_S7/ATTN_B1_2021_11_05_13_06_40_chunk3.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021NES/output_2021_11_05_NES_S9/ATTN_B3_2021_11_05_13_06_59_chunk3.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021SLU/output_2021_06_10_SLU_S2/ATTN_bot2p2_Zoom_Meeting_Room_2_2021_06_10_12_37_25_chunk6.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2021SLU/output_2021_06_10_SLU_S3/ATTN_bot3p2_Room_3_Zoom_Meeting_2021_

Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2022MND/output_2022_04_08_MND_S8/ATTN_Bot2_Zoom_Meeting_Room_2_2022_04_08_13_02_14_chunk1.json (Expecting value: line 1 column 1 (char 0))
Skipping invalid JSON file: /Users/maxchalekson/Projects/gemini_data_analysis/outputs/2022MND/output_2022_04_08_MND_S9/ATTN_Bot3_2022_04_08_13_01_56_problem_55min30sec_57min20sec_chunk1.json (Expecting value: line 1 column 1 (char 0))
Skipped 39 invalid output JSON files.


(  conference  normalized_name output_raw_names  \
 0    2020NES  adam holewinski  Adam Holewinski   
 1    2020NES      alissa park      Alissa Park   
 2    2020NES     andrea hicks     Andrea Hicks   
 3    2020NES           andrew           Andrew   
 4    2020NES      andrew feig      Andrew Feig   
 
                                       output_sources  \
 0  chunk_summary.speaking_time_seconds | session_...   
 1  chunk_summary.speaking_time_seconds | session_...   
 2  chunk_summary.speaking_time_seconds | session_...   
 3  chunk_summary.speaking_time_seconds | session_...   
 4  chunk_summary.speaking_time_seconds | session_...   
 
                                  output_source_files  output_occurrence_count  \
 0  outputs/2020NES/output_2020_11_05_NES_S4/4_Bey...                       92   
 1  outputs/2020NES/output_2020_11_05_NES_S1/1_DAC...                      140   
 2  outputs/2020NES/output_2020_11_05_NES_S5/5_Dec...                       49   
 3  outputs/2020NES/

In [3]:
matched_df = output_df.merge(
    truth_df,
    on=["conference", "normalized_name"],
    how="inner",
    suffixes=("_output", "_truth"),
)

unmatched_output_df = output_df.merge(
    truth_df[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_output_df = unmatched_output_df[unmatched_output_df["_merge"] == "left_only"].drop(columns=["_merge"])

unmatched_truth_df = truth_df.merge(
    output_df[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_truth_df = unmatched_truth_df[unmatched_truth_df["_merge"] == "left_only"].drop(columns=["_merge"])

conference_order = sorted(set(output_df["conference"].unique()).union(truth_df["conference"].unique())) if not output_df.empty or not truth_df.empty else []
progress_rows = []

for conference in conference_order:
    output_conference = output_df[output_df["conference"] == conference] if not output_df.empty else pd.DataFrame()
    truth_conference = truth_df[truth_df["conference"] == conference] if not truth_df.empty else pd.DataFrame()
    matched_conference = matched_df[matched_df["conference"] == conference] if not matched_df.empty else pd.DataFrame()

    output_unique = len(output_conference)
    truth_unique = len(truth_conference)
    matched_unique = len(matched_conference)

    progress_rows.append({
        "conference": conference,
        "output_unique_participants": output_unique,
        "truth_unique_participants": truth_unique,
        "matched_participants": matched_unique,
        "unmatched_output_participants": output_unique - matched_unique,
        "unmatched_truth_participants": truth_unique - matched_unique,
        "output_match_rate": round(matched_unique / output_unique, 4) if output_unique else 0.0,
        "truth_match_rate": round(matched_unique / truth_unique, 4) if truth_unique else 0.0,
    })

progress_df = pd.DataFrame(progress_rows).sort_values("conference").reset_index(drop=True) if progress_rows else pd.DataFrame(
    columns=[
        "conference",
        "output_unique_participants",
        "truth_unique_participants",
        "matched_participants",
        "unmatched_output_participants",
        "unmatched_truth_participants",
        "output_match_rate",
        "truth_match_rate",
    ]
)

matched_output_path = BASE / "matched_output_participants.csv"
unmatched_output_path = BASE / "unmatched_output_participants.csv"
unmatched_truth_path = BASE / "unmatched_truth_participants.csv"
progress_path = BASE / "participant_matching_progress_by_conference.csv"

matched_df.to_csv(matched_output_path, index=False)
unmatched_output_df.to_csv(unmatched_output_path, index=False)
unmatched_truth_df.to_csv(unmatched_truth_path, index=False)
progress_df.to_csv(progress_path, index=False)

print(f"Saved: {matched_output_path}")
print(f"Saved: {unmatched_output_path}")
print(f"Saved: {unmatched_truth_path}")
print(f"Saved: {progress_path}")

print(f"Output participants: {len(output_df)}")
print(f"Truth participants: {len(truth_df)}")
print(f"Matched participants: {len(matched_df)}")

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/matched_output_participants.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_output_participants.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_truth_participants.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_matching_progress_by_conference.csv
Output participants: 762
Truth participants: 686
Matched participants: 648


In [4]:
def build_unmatched_review(df, label):
    if df.empty:
        return pd.DataFrame(
            columns=[
                "normalized_name",
                "record_type",
                "conferences",
                "raw_names",
                "source_files",
                "occurrence_count",
                "unique_file_count",
            ]
        )

    raw_name_column = f"{label}_raw_names"
    source_file_column = f"{label}_source_files"
    occurrence_column = f"{label}_occurrence_count"

    review_df = (
        df.assign(
            raw_name=df[raw_name_column],
            source_file=df[source_file_column],
            occurrence=df[occurrence_column],
            record_type=label,
        )
        .groupby("normalized_name", as_index=False)
        .agg(
            record_type=("record_type", lambda values: " | ".join(sorted(set(values)))),
            conferences=("conference", lambda values: " | ".join(sorted(set(values)))),
            raw_names=("raw_name", lambda values: " | ".join(sorted(set(filter(None, values))))),
            source_files=("source_file", lambda values: " | ".join(sorted(set(filter(None, values))))),
            occurrence_count=("occurrence", "sum"),
        )
    )
    review_df["unique_file_count"] = review_df["source_files"].apply(lambda value: 0 if not value else len(value.split(" | ")))
    return review_df.sort_values(["conferences", "normalized_name"]).reset_index(drop=True)

unmatched_output_review_df = build_unmatched_review(unmatched_output_df, "output")
unmatched_truth_review_df = build_unmatched_review(unmatched_truth_df, "truth")

unmatched_review_df = (
    pd.concat([unmatched_output_review_df, unmatched_truth_review_df], ignore_index=True)
    .sort_values(["conferences", "normalized_name"])
    .reset_index(drop=True)
    if not unmatched_output_review_df.empty or not unmatched_truth_review_df.empty
    else pd.DataFrame(
        columns=[
            "normalized_name",
            "record_type",
            "conferences",
            "raw_names",
            "source_files",
            "occurrence_count",
            "unique_file_count",
        ]
    )
)

unmatched_review_path = BASE / "unmatched_name_review.csv"
unmatched_review_df.to_csv(unmatched_review_path, index=False)
print(f"Saved: {unmatched_review_path}")

def build_manual_alias_template(review_df):
    if review_df.empty:
        return pd.DataFrame(
            columns=[
                "alias_name",
                "canonical_name",
                "record_type",
                "conferences",
                "raw_names",
                "source_files",
                "notes",
            ]
        )

    template_df = review_df.copy()
    template_df = template_df.rename(columns={"normalized_name": "alias_name"})
    template_df["canonical_name"] = ""
    template_df["notes"] = ""
    template_df = template_df[[
        "alias_name",
        "canonical_name",
        "record_type",
        "conferences",
        "raw_names",
        "source_files",
        "notes",
    ]]
    return template_df

manual_alias_template_path = BASE / "participant_alias_mapping_template.csv"
if not manual_alias_template_path.exists():
    manual_alias_template_df = build_manual_alias_template(unmatched_review_df)
    manual_alias_template_df.to_csv(manual_alias_template_path, index=False)
    print(f"Saved: {manual_alias_template_path}")
else:
    print(f"Template already exists: {manual_alias_template_path}")

print("If you want the notebook to use manual aliases on the next run, create or edit:")
print(f"- {REPO_ROOT / 'analysis_v1' / 'participant_alias_mapping.csv'}")
print("Use the template as a starting point.")

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_name_review.csv
Template already exists: /Users/maxchalekson/Projects/gemini_data_analysis/participant_alias_mapping_template.csv
If you want the notebook to use manual aliases on the next run, create or edit:
- /Users/maxchalekson/Projects/gemini_data_analysis/analysis_v1/participant_alias_mapping.csv
Use the template as a starting point.


In [5]:
display_columns = [
    "conference",
    "normalized_name",
    "output_raw_names",
    "truth_raw_names",
]

print("Matched participants by conference:")
display(progress_df)

print("Unmatched name review sample:")
display(unmatched_review_df.head(20))

print("Matched participant sample:")
display(matched_df[display_columns].head(20) if not matched_df.empty else matched_df)

Matched participants by conference:


,conference,output_unique_participants,truth_unique_participants,matched_participants,unmatched_output_participants,unmatched_truth_participants,output_match_rate,truth_match_rate
0,2020NES,110,94,94,16,0,0.8545,1.0000
1,2021ABI,138,117,110,28,7,0.7971,0.9402
2,2021CMC,84,77,72,12,5,0.8571,0.9351
3,2021MND,107,98,96,11,2,0.8972,0.9796
4,2021MZT,81,73,60,21,13,0.7407,0.8219
5,2021NES,96,92,88,8,4,0.9167,0.9565
6,2021SLU,88,81,74,14,7,0.8409,0.9136
7,2022MND,58,54,54,4,0,0.9310,1.0000


Unmatched name review sample:


,normalized_name,record_type,conferences,raw_names,source_files,occurrence_count,unique_file_count
0,cheng liu,output,2020NES,Cheng Liu,outputs/2020NES/output_2020_11_06_NES_S3/2020_...,6,4
1,christopher gorski,output,2020NES,Christopher Gorski,outputs/2020NES/output_2020_11_05_NES_S1/1_DAC...,34,14
2,david gator kwab,output,2020NES,David Gator Kwab,outputs/2020NES/output_2020_11_06_NES_S12/6_ne...,9,3
3,jarad mason,output,2020NES,Jarad Mason,outputs/2020NES/output_2020_11_06_NES_S1/1_CO2...,40,15
4,lindsey seitz,output,2020NES,Lindsey Seitz,outputs/2020NES/output_2020_11_05_NES_S6/6_The...,4,2
5,shaama mallikarjun sharada,output,2020NES,Shaama Mallikarjun Sharada,outputs/2020NES/output_2020_11_05_NES_S6/6_The...,3,1
6,shama m sharada,output,2020NES,Shama M. Sharada,outputs/2020NES/output_2020_11_06_NES_S3/2020_...,12,4
7,simona niguari,output,2020NES,Simona Niguari,outputs/2020NES/output_2020_11_06_NES_S9/2020_...,18,8
8,speaker 1,output,2020NES,Speaker 1,outputs/2020NES/output_2020_11_05_NES_S4/4_Bey...,9,3
9,speaker 2,output,2020NES,Speaker 2,outputs/2020NES/output_2020_11_06_NES_S7/Zoom_...,3,1


Matched participant sample:


,conference,normalized_name,output_raw_names,truth_raw_names
0,2020NES,adam holewinski,Adam Holewinski,Adam Holewinski
1,2020NES,alissa park,Alissa Park,alissa park
2,2020NES,andrea hicks,Andrea Hicks,Andrea Hicks
3,2020NES,andrew,Andrew,andrew
4,2020NES,andrew feig,Andrew Feig,andrew feig
5,2020NES,andrew teixeira,Andrew Teixeira,Andrew Teixeira
6,2020NES,angela hagen,Angela Hagen,angela hagen
7,2020NES,ashleigh baber,Ashleigh Baber,ashleigh baber
8,2020NES,betar gallant,Betar Gallant,Betar Gallant
9,2020NES,betsy cantwell,Betsy Cantwell,betsy cantwell


## Auto-resolve remaining unmatched participants

This section uses conference-constrained fuzzy name matching to propose and auto-apply high-confidence alias links from unmatched output names to unmatched truth names.

It writes:
- `auto_alias_suggestions.csv` (all ranked candidates)
- `auto_alias_accepted.csv` (auto-accepted links)
- `matched_output_participants_final.csv`
- `unmatched_output_participants_final.csv`
- `unmatched_truth_participants_final.csv`
- `participant_matching_progress_by_conference_final.csv`

You can optionally move accepted aliases into `participant_alias_mapping.csv` for future runs.

In [6]:
from difflib import SequenceMatcher

# Conservative defaults: prioritize precision over recall for auto-accept.
AUTO_ACCEPT_SCORE = 0.92
MIN_CANDIDATE_SCORE = 0.65


def _tokens(name: str):
    return [t for t in str(name).split() if t]


def _signature(name: str):
    tokens = _tokens(name)
    if not tokens:
        return "", "", ""
    first = tokens[0]
    last = tokens[-1]
    first_initial = first[0] if first else ""
    return first, last, first_initial


def _name_similarity(a: str, b: str) -> float:
    a = str(a).strip()
    b = str(b).strip()
    if not a or not b:
        return 0.0

    if a == b:
        return 1.0

    a_first, a_last, a_init = _signature(a)
    b_first, b_last, b_init = _signature(b)

    base = SequenceMatcher(None, a, b).ratio()
    first_ratio = SequenceMatcher(None, a_first, b_first).ratio() if a_first and b_first else 0.0
    last_ratio = SequenceMatcher(None, a_last, b_last).ratio() if a_last and b_last else 0.0

    # Strongly weight surname agreement and first-name similarity.
    score = (0.45 * last_ratio) + (0.35 * first_ratio) + (0.20 * base)

    # Bonus when initials align.
    if a_init and b_init and a_init == b_init:
        score += 0.03

    return min(score, 1.0)


def build_alias_suggestions(unmatched_output_df, unmatched_truth_df, top_k=5):
    if unmatched_output_df.empty or unmatched_truth_df.empty:
        return pd.DataFrame(
            columns=[
                "conference",
                "output_normalized_name",
                "truth_normalized_name",
                "similarity_score",
                "output_raw_names",
                "truth_raw_names",
                "rank",
            ]
        )

    rows = []

    for conference in sorted(set(unmatched_output_df["conference"]) & set(unmatched_truth_df["conference"])):
        output_names = unmatched_output_df[unmatched_output_df["conference"] == conference]
        truth_names = unmatched_truth_df[unmatched_truth_df["conference"] == conference]

        for _, out_row in output_names.iterrows():
            out_name = out_row["normalized_name"]
            candidates = []

            for _, truth_row in truth_names.iterrows():
                truth_name = truth_row["normalized_name"]
                score = _name_similarity(out_name, truth_name)
                if score >= MIN_CANDIDATE_SCORE:
                    candidates.append((score, truth_name, truth_row))

            candidates.sort(key=lambda item: item[0], reverse=True)
            for rank, (score, truth_name, truth_row) in enumerate(candidates[:top_k], start=1):
                rows.append(
                    {
                        "conference": conference,
                        "output_normalized_name": out_name,
                        "truth_normalized_name": truth_name,
                        "similarity_score": round(float(score), 4),
                        "output_raw_names": out_row.get("output_raw_names", ""),
                        "truth_raw_names": truth_row.get("truth_raw_names", ""),
                        "rank": rank,
                    }
                )

    if not rows:
        return pd.DataFrame(
            columns=[
                "conference",
                "output_normalized_name",
                "truth_normalized_name",
                "similarity_score",
                "output_raw_names",
                "truth_raw_names",
                "rank",
            ]
        )

    return pd.DataFrame(rows).sort_values(
        ["conference", "output_normalized_name", "rank"], ascending=[True, True, True]
    ).reset_index(drop=True)


def choose_auto_aliases(suggestions_df):
    if suggestions_df.empty:
        return pd.DataFrame(columns=["conference", "alias_name", "canonical_name", "similarity_score", "status"])

    top = suggestions_df[suggestions_df["rank"] == 1].copy()
    if top.empty:
        return pd.DataFrame(columns=["conference", "alias_name", "canonical_name", "similarity_score", "status"])

    # Keep only high-confidence first-choice links.
    top = top[top["similarity_score"] >= AUTO_ACCEPT_SCORE].copy()
    if top.empty:
        return pd.DataFrame(columns=["conference", "alias_name", "canonical_name", "similarity_score", "status"])

    # One-to-one constraint: keep unique alias and canonical within each conference.
    top = top.sort_values(["conference", "similarity_score"], ascending=[True, False])
    top = top.drop_duplicates(subset=["conference", "output_normalized_name"], keep="first")
    top = top.drop_duplicates(subset=["conference", "truth_normalized_name"], keep="first")

    accepted = top.rename(
        columns={
            "output_normalized_name": "alias_name",
            "truth_normalized_name": "canonical_name",
        }
    )[["conference", "alias_name", "canonical_name", "similarity_score"]]
    accepted["status"] = "auto_accepted"
    return accepted.reset_index(drop=True)


alias_suggestions_df = build_alias_suggestions(unmatched_output_df, unmatched_truth_df, top_k=5)
auto_alias_df = choose_auto_aliases(alias_suggestions_df)

suggestions_path = BASE / "auto_alias_suggestions.csv"
accepted_path = BASE / "auto_alias_accepted.csv"
alias_suggestions_df.to_csv(suggestions_path, index=False)
auto_alias_df.to_csv(accepted_path, index=False)

print(f"Saved: {suggestions_path}")
print(f"Saved: {accepted_path}")
print(f"Suggested candidate rows: {len(alias_suggestions_df)}")
print(f"Auto-accepted aliases: {len(auto_alias_df)}")

display(alias_suggestions_df.head(30))

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/auto_alias_suggestions.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/auto_alias_accepted.csv
Suggested candidate rows: 39
Auto-accepted aliases: 21


,conference,output_normalized_name,truth_normalized_name,similarity_score,output_raw_names,truth_raw_names,rank
0,2021ABI,alex walsh,alexandra walsh,0.8554,Alex Walsh,Alexandra Walsh,1
1,2021ABI,beck kamilov,ulugbek kamilov,0.8039,Beck Kamilov,Ulugbek Kamilov,1
2,2021ABI,gokul upadhyayula,srigokul upadhyayula,0.9030,Gokul Upadhyayula,Srigokul Upadhyayula,1
3,2021ABI,josh brake,joshua brake,0.9418,Josh Brake,Joshua Brake,1
4,2021ABI,seu sim,seunghyun sim,0.7950,Seu Sim,Seunghyun Sim,1
5,2021ABI,shiva abbaszadeh,shiva abbbaszadeh,1.0000,Shiva Abbaszadeh,Shiva Abbbaszadeh,1
6,2021CMC,alex green,alexander green,0.8554,Alex Green,Alexander Green,1
7,2021CMC,gw gant luxton,g w gant luxton,0.9064,GW Gant Luxton,G.W. Gant Luxton,1
8,2021CMC,macha kamenetska,maria kamenetska,0.8650,Macha Kamenetska,Maria (Masha) Kamenetska,1
9,2021CMC,maisha kamenetska,maria kamenetska,0.9164,Maisha Kamenetska,Maria (Masha) Kamenetska,1


In [7]:
def apply_auto_aliases(df, alias_df):
    if df.empty or alias_df.empty:
        return df.copy()

    result = df.copy()
    conference_alias_map = {
        conf: dict(zip(part["alias_name"], part["canonical_name"]))
        for conf, part in alias_df.groupby("conference")
    }

    def _map_name(row):
        conf = row["conference"]
        name = row["normalized_name"]
        mapping = conference_alias_map.get(conf, {})
        return mapping.get(name, name)

    result["normalized_name"] = result.apply(_map_name, axis=1)

    # Re-aggregate rows collapsed by aliasing so counts and provenance remain additive.
    result = (
        result.groupby(["conference", "normalized_name"], as_index=False)
        .agg(
            output_raw_names=("output_raw_names", lambda v: " | ".join(sorted(set(" | ".join(v).split(" | "))))),
            output_sources=("output_sources", lambda v: " | ".join(sorted(set(" | ".join(v).split(" | "))))),
            output_source_files=("output_source_files", lambda v: " | ".join(sorted(set(" | ".join(v).split(" | "))))),
            output_occurrence_count=("output_occurrence_count", "sum"),
            output_unique_file_count=("output_unique_file_count", "sum"),
        )
        .sort_values(["conference", "normalized_name"])
        .reset_index(drop=True)
    )

    return result


output_df_final = apply_auto_aliases(output_df, auto_alias_df)

matched_final_df = output_df_final.merge(
    truth_df,
    on=["conference", "normalized_name"],
    how="inner",
)

unmatched_output_final_df = output_df_final.merge(
    truth_df[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_output_final_df = unmatched_output_final_df[unmatched_output_final_df["_merge"] == "left_only"].drop(columns=["_merge"])

unmatched_truth_final_df = truth_df.merge(
    output_df_final[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_truth_final_df = unmatched_truth_final_df[unmatched_truth_final_df["_merge"] == "left_only"].drop(columns=["_merge"])

conference_order_final = sorted(
    set(output_df_final["conference"].unique()).union(truth_df["conference"].unique())
)
progress_rows_final = []
for conference in conference_order_final:
    out_c = output_df_final[output_df_final["conference"] == conference]
    truth_c = truth_df[truth_df["conference"] == conference]
    matched_c = matched_final_df[matched_final_df["conference"] == conference]

    output_unique = len(out_c)
    truth_unique = len(truth_c)
    matched_unique = len(matched_c)

    progress_rows_final.append(
        {
            "conference": conference,
            "output_unique_participants": output_unique,
            "truth_unique_participants": truth_unique,
            "matched_participants": matched_unique,
            "unmatched_output_participants": output_unique - matched_unique,
            "unmatched_truth_participants": truth_unique - matched_unique,
            "output_match_rate": round(matched_unique / output_unique, 4) if output_unique else 0.0,
            "truth_match_rate": round(matched_unique / truth_unique, 4) if truth_unique else 0.0,
        }
    )

progress_final_df = pd.DataFrame(progress_rows_final).sort_values("conference").reset_index(drop=True)

matched_final_path = BASE / "matched_output_participants_final.csv"
unmatched_output_final_path = BASE / "unmatched_output_participants_final.csv"
unmatched_truth_final_path = BASE / "unmatched_truth_participants_final.csv"
progress_final_path = BASE / "participant_matching_progress_by_conference_final.csv"

matched_final_df.to_csv(matched_final_path, index=False)
unmatched_output_final_df.to_csv(unmatched_output_final_path, index=False)
unmatched_truth_final_df.to_csv(unmatched_truth_final_path, index=False)
progress_final_df.to_csv(progress_final_path, index=False)

print(f"Saved: {matched_final_path}")
print(f"Saved: {unmatched_output_final_path}")
print(f"Saved: {unmatched_truth_final_path}")
print(f"Saved: {progress_final_path}")

print("\nBefore auto-aliases:")
print(f"- unmatched output: {len(unmatched_output_df)}")
print(f"- unmatched truth: {len(unmatched_truth_df)}")

print("\nAfter auto-aliases:")
print(f"- unmatched output: {len(unmatched_output_final_df)}")
print(f"- unmatched truth: {len(unmatched_truth_final_df)}")

display(progress_final_df)

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/matched_output_participants_final.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_output_participants_final.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_truth_participants_final.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_matching_progress_by_conference_final.csv

Before auto-aliases:
- unmatched output: 114
- unmatched truth: 38

After auto-aliases:
- unmatched output: 93
- unmatched truth: 17


,conference,output_unique_participants,truth_unique_participants,matched_participants,unmatched_output_participants,unmatched_truth_participants,output_match_rate,truth_match_rate
0,2020NES,110,94,94,16,0,0.8545,1.0000
1,2021ABI,138,117,112,26,5,0.8116,0.9573
2,2021CMC,84,77,72,12,5,0.8571,0.9351
3,2021MND,107,98,96,11,2,0.8972,0.9796
4,2021MZT,81,73,71,10,2,0.8765,0.9726
5,2021NES,96,92,89,7,3,0.9271,0.9674
6,2021SLU,88,81,81,7,0,0.9205,1.0000
7,2022MND,58,54,54,4,0,0.9310,1.0000


## Manual resolution pass for remaining truth unmatched

Use this cell to apply curated aliases for the difficult remainder after auto-matching.

It writes:
- `matched_output_participants_reviewed.csv`
- `unmatched_output_participants_reviewed.csv`
- `unmatched_truth_participants_reviewed.csv`
- `participant_matching_progress_by_conference_reviewed.csv`

Edit `MANUAL_REMAINING_ALIAS_MAP` as needed, then rerun this cell.

In [8]:
# Curated aliases from remaining unmatched truth names.
# Left side is unmatched output normalized_name, right side is truth normalized_name.
MANUAL_REMAINING_ALIAS_MAP = {
    "2021ABI": {
        "alex walsh": "alexandra walsh",
        "doug shepherd": "douglas shepherd",
        "seu sim": "seunghyun sim",
        "gokul upadhyayula": "srigokul upadhyayula",
        "beck kamilov": "ulugbek kamilov",
    },
    "2021CMC": {
        "alex green": "alexander green",
        "gw gant luxton": "g w gant luxton",
        "masha kamenetska": "maria kamenetska",
        "ross wang": "rongsheng wang",
        "stephen yi": "s stephen yi",
    },
    "2021MND": {
        "jp yu": "john paul yu",
        "calvin ye": "kaixiong ye",
    },
    "2021MZT": {
        "matt hopken": "matthew w hopken",
        "becky smith": "rebecca l smith",
    },
    "2021NES": {
        "leslie abdul aziz": "kandis gilliard abdul aziz",
        "leo liu": "t leo liu",
        "will bowman": "william j bowman",
    },
    "2022MND": {
        "dave durgan": "david durgan",
    },
}

manual_rows = []
for conf, mapping in MANUAL_REMAINING_ALIAS_MAP.items():
    for alias_name, canonical_name in mapping.items():
        manual_rows.append(
            {
                "conference": conf,
                "alias_name": alias_name,
                "canonical_name": canonical_name,
                "similarity_score": None,
                "status": "manual_remaining_resolution",
            }
        )

manual_remaining_alias_df = pd.DataFrame(manual_rows)

if auto_alias_df.empty:
    alias_union_df = manual_remaining_alias_df.copy()
else:
    alias_union_df = pd.concat([auto_alias_df, manual_remaining_alias_df], ignore_index=True)

# Last write wins so manual overrides can replace earlier choices.
alias_union_df = alias_union_df.drop_duplicates(
    subset=["conference", "alias_name"], keep="last"
).reset_index(drop=True)

output_df_reviewed = apply_auto_aliases(output_df, alias_union_df)

matched_reviewed_df = output_df_reviewed.merge(
    truth_df,
    on=["conference", "normalized_name"],
    how="inner",
)

unmatched_output_reviewed_df = output_df_reviewed.merge(
    truth_df[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_output_reviewed_df = unmatched_output_reviewed_df[
    unmatched_output_reviewed_df["_merge"] == "left_only"
].drop(columns=["_merge"])

unmatched_truth_reviewed_df = truth_df.merge(
    output_df_reviewed[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_truth_reviewed_df = unmatched_truth_reviewed_df[
    unmatched_truth_reviewed_df["_merge"] == "left_only"
].drop(columns=["_merge"])

conference_order_reviewed = sorted(
    set(output_df_reviewed["conference"].unique()).union(truth_df["conference"].unique())
)
progress_rows_reviewed = []
for conference in conference_order_reviewed:
    out_c = output_df_reviewed[output_df_reviewed["conference"] == conference]
    truth_c = truth_df[truth_df["conference"] == conference]
    matched_c = matched_reviewed_df[matched_reviewed_df["conference"] == conference]

    output_unique = len(out_c)
    truth_unique = len(truth_c)
    matched_unique = len(matched_c)

    progress_rows_reviewed.append(
        {
            "conference": conference,
            "output_unique_participants": output_unique,
            "truth_unique_participants": truth_unique,
            "matched_participants": matched_unique,
            "unmatched_output_participants": output_unique - matched_unique,
            "unmatched_truth_participants": truth_unique - matched_unique,
            "output_match_rate": round(matched_unique / output_unique, 4) if output_unique else 0.0,
            "truth_match_rate": round(matched_unique / truth_unique, 4) if truth_unique else 0.0,
        }
    )

progress_reviewed_df = pd.DataFrame(progress_rows_reviewed).sort_values("conference").reset_index(drop=True)

matched_reviewed_path = BASE / "matched_output_participants_reviewed.csv"
unmatched_output_reviewed_path = BASE / "unmatched_output_participants_reviewed.csv"
unmatched_truth_reviewed_path = BASE / "unmatched_truth_participants_reviewed.csv"
progress_reviewed_path = BASE / "participant_matching_progress_by_conference_reviewed.csv"

matched_reviewed_df.to_csv(matched_reviewed_path, index=False)
unmatched_output_reviewed_df.to_csv(unmatched_output_reviewed_path, index=False)
unmatched_truth_reviewed_df.to_csv(unmatched_truth_reviewed_path, index=False)
progress_reviewed_df.to_csv(progress_reviewed_path, index=False)

print(f"Saved: {matched_reviewed_path}")
print(f"Saved: {unmatched_output_reviewed_path}")
print(f"Saved: {unmatched_truth_reviewed_path}")
print(f"Saved: {progress_reviewed_path}")

print("\nAfter reviewed manual pass:")
print(f"- unmatched output: {len(unmatched_output_reviewed_df)}")
print(f"- unmatched truth: {len(unmatched_truth_reviewed_df)}")

if not unmatched_truth_reviewed_df.empty:
    print("\nStill unmatched truth names:")
    display(unmatched_truth_reviewed_df[["conference", "normalized_name", "truth_raw_names"]].sort_values(["conference", "normalized_name"]))

# Save combined alias map for persistence.
alias_union_path = BASE / "participant_alias_mapping_reviewed.csv"
alias_union_df[["conference", "alias_name", "canonical_name", "status"]].to_csv(alias_union_path, index=False)
print(f"Saved: {alias_union_path}")

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/matched_output_participants_reviewed.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_output_participants_reviewed.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_truth_participants_reviewed.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_matching_progress_by_conference_reviewed.csv

After reviewed manual pass:
- unmatched output: 76
- unmatched truth: 1

Still unmatched truth names:


,conference,normalized_name,truth_raw_names
98,2021ABI,alexandra dickinson,Alexandra Dickinson


Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_alias_mapping_reviewed.csv


In [9]:
# Final targeted patch for the last remaining truth name in 2021ABI.
FINAL_ALIAS_PATCH = {
    "2021ABI": {
        "jazz dickinson": "alexandra dickinson",
        "jazz dickison": "alexandra dickinson",
    }
}

extra_rows = []
for conf, mapping in FINAL_ALIAS_PATCH.items():
    for alias_name, canonical_name in mapping.items():
        extra_rows.append(
            {
                "conference": conf,
                "alias_name": alias_name,
                "canonical_name": canonical_name,
                "similarity_score": None,
                "status": "manual_final_patch",
            }
        )

extra_alias_df = pd.DataFrame(extra_rows)
alias_union_final_df = pd.concat([alias_union_df, extra_alias_df], ignore_index=True)
alias_union_final_df = alias_union_final_df.drop_duplicates(
    subset=["conference", "alias_name"], keep="last"
).reset_index(drop=True)

output_df_final_review = apply_auto_aliases(output_df, alias_union_final_df)

matched_final_review_df = output_df_final_review.merge(
    truth_df,
    on=["conference", "normalized_name"],
    how="inner",
)

unmatched_output_final_review_df = output_df_final_review.merge(
    truth_df[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_output_final_review_df = unmatched_output_final_review_df[
    unmatched_output_final_review_df["_merge"] == "left_only"
].drop(columns=["_merge"])

unmatched_truth_final_review_df = truth_df.merge(
    output_df_final_review[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_truth_final_review_df = unmatched_truth_final_review_df[
    unmatched_truth_final_review_df["_merge"] == "left_only"
].drop(columns=["_merge"])

progress_rows_final_review = []
conference_order_final_review = sorted(
    set(output_df_final_review["conference"].unique()).union(truth_df["conference"].unique())
)
for conference in conference_order_final_review:
    out_c = output_df_final_review[output_df_final_review["conference"] == conference]
    truth_c = truth_df[truth_df["conference"] == conference]
    matched_c = matched_final_review_df[matched_final_review_df["conference"] == conference]

    output_unique = len(out_c)
    truth_unique = len(truth_c)
    matched_unique = len(matched_c)

    progress_rows_final_review.append(
        {
            "conference": conference,
            "output_unique_participants": output_unique,
            "truth_unique_participants": truth_unique,
            "matched_participants": matched_unique,
            "unmatched_output_participants": output_unique - matched_unique,
            "unmatched_truth_participants": truth_unique - matched_unique,
            "output_match_rate": round(matched_unique / output_unique, 4) if output_unique else 0.0,
            "truth_match_rate": round(matched_unique / truth_unique, 4) if truth_unique else 0.0,
        }
    )

progress_final_review_df = pd.DataFrame(progress_rows_final_review).sort_values("conference").reset_index(drop=True)

matched_final_review_path = BASE / "matched_output_participants_final_review.csv"
unmatched_output_final_review_path = BASE / "unmatched_output_participants_final_review.csv"
unmatched_truth_final_review_path = BASE / "unmatched_truth_participants_final_review.csv"
progress_final_review_path = BASE / "participant_matching_progress_by_conference_final_review.csv"
alias_union_final_path = BASE / "participant_alias_mapping_final_review.csv"

matched_final_review_df.to_csv(matched_final_review_path, index=False)
unmatched_output_final_review_df.to_csv(unmatched_output_final_review_path, index=False)
unmatched_truth_final_review_df.to_csv(unmatched_truth_final_review_path, index=False)
progress_final_review_df.to_csv(progress_final_review_path, index=False)
alias_union_final_df[["conference", "alias_name", "canonical_name", "status"]].to_csv(alias_union_final_path, index=False)

print(f"Saved: {matched_final_review_path}")
print(f"Saved: {unmatched_output_final_review_path}")
print(f"Saved: {unmatched_truth_final_review_path}")
print(f"Saved: {progress_final_review_path}")
print(f"Saved: {alias_union_final_path}")

print("\nAfter final patch:")
print(f"- unmatched output: {len(unmatched_output_final_review_df)}")
print(f"- unmatched truth: {len(unmatched_truth_final_review_df)}")

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/matched_output_participants_final_review.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_output_participants_final_review.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_truth_participants_final_review.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_matching_progress_by_conference_final_review.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_alias_mapping_final_review.csv

After final patch:
- unmatched output: 74
- unmatched truth: 0


## Conference-by-conference mismatch resolver (pass 2)

This pass attempts to resolve remaining output mismatches against each conference truth roster using stricter rules:
- Prefer exact surname agreement.
- Allow nickname/initial variants for first names.
- Require a clear score gap from the second-best candidate.
- Skip generic placeholders like `unknown speaker`, `speaker 1`, etc.

Outputs:
- `conference_pass2_alias_suggestions.csv`
- `conference_pass2_alias_accepted.csv`
- `matched_output_participants_conference_resolved.csv`
- `unmatched_output_participants_conference_resolved.csv`
- `unmatched_truth_participants_conference_resolved.csv`
- `participant_matching_progress_by_conference_conference_resolved.csv`

In [10]:
from difflib import SequenceMatcher

NICKNAME_MAP = {
    "alex": "alexandra",
    "doug": "douglas",
    "dave": "david",
    "matt": "matthew",
    "becky": "rebecca",
    "bill": "william",
    "will": "william",
    "jim": "james",
    "bob": "robert",
    "kathy": "katherine",
    "kate": "katherine",
}

GENERIC_NAME_TOKENS = {
    "unknown",
    "speaker",
    "participant",
    "moderator",
    "panelist",
    "host",
    "guest",
}

PASS2_STRONG_THRESHOLD = 0.90
PASS2_NICKNAME_THRESHOLD = 0.86
PASS2_MARGIN = 0.06


def _name_parts(name: str):
    toks = [t for t in str(name).split() if t]
    if not toks:
        return "", "", "", toks
    first = toks[0]
    last = toks[-1]
    initial = first[0] if first else ""
    return first, last, initial, toks


def _is_placeholder_name(name: str) -> bool:
    name = str(name).strip().lower()
    if not name:
        return True
    if re.fullmatch(r"speaker\s*\d+", name):
        return True
    toks = set(name.split())
    return bool(toks & GENERIC_NAME_TOKENS)


def _first_name_equiv(a: str, b: str) -> bool:
    a = str(a)
    b = str(b)
    if a == b:
        return True
    if NICKNAME_MAP.get(a) == b or NICKNAME_MAP.get(b) == a:
        return True
    return False


def _pass2_score(out_name: str, truth_name: str):
    of, ol, oi, _ = _name_parts(out_name)
    tf, tl, ti, _ = _name_parts(truth_name)
    if not ol or not tl:
        return 0.0, False, False

    last_exact = ol == tl
    first_equiv = _first_name_equiv(of, tf) or (oi and ti and oi == ti)

    full_ratio = SequenceMatcher(None, out_name, truth_name).ratio()
    first_ratio = SequenceMatcher(None, of, tf).ratio() if of and tf else 0.0
    last_ratio = SequenceMatcher(None, ol, tl).ratio()

    score = (0.50 * last_ratio) + (0.30 * first_ratio) + (0.20 * full_ratio)
    if last_exact:
        score += 0.05
    if first_equiv:
        score += 0.05
    return min(score, 1.0), last_exact, first_equiv


def build_pass2_suggestions(unmatched_output_df, truth_df, top_k=3):
    rows = []
    common_conferences = sorted(set(unmatched_output_df["conference"]) & set(truth_df["conference"]))

    for conf in common_conferences:
        out_conf = unmatched_output_df[unmatched_output_df["conference"] == conf]
        truth_conf = truth_df[truth_df["conference"] == conf]

        for _, out_row in out_conf.iterrows():
            out_name = out_row["normalized_name"]
            if _is_placeholder_name(out_name):
                continue

            candidates = []
            for _, truth_row in truth_conf.iterrows():
                truth_name = truth_row["normalized_name"]
                score, last_exact, first_equiv = _pass2_score(out_name, truth_name)
                if score >= 0.65:
                    candidates.append((score, last_exact, first_equiv, truth_row))

            if not candidates:
                continue

            candidates.sort(key=lambda x: x[0], reverse=True)
            top = candidates[:top_k]
            for rank, (score, last_exact, first_equiv, truth_row) in enumerate(top, start=1):
                rows.append(
                    {
                        "conference": conf,
                        "output_normalized_name": out_name,
                        "truth_normalized_name": truth_row["normalized_name"],
                        "score": round(float(score), 4),
                        "last_name_exact": bool(last_exact),
                        "first_name_equiv": bool(first_equiv),
                        "rank": rank,
                        "output_occurrence_count": out_row.get("output_occurrence_count", 0),
                        "output_raw_names": out_row.get("output_raw_names", ""),
                        "truth_raw_names": truth_row.get("truth_raw_names", ""),
                    }
                )

    if not rows:
        return pd.DataFrame(
            columns=[
                "conference", "output_normalized_name", "truth_normalized_name", "score", "last_name_exact",
                "first_name_equiv", "rank", "output_occurrence_count", "output_raw_names", "truth_raw_names"
            ]
        )

    return pd.DataFrame(rows).sort_values(
        ["conference", "output_normalized_name", "rank"],
        ascending=[True, True, True],
    ).reset_index(drop=True)


def accept_pass2_aliases(suggestions_df):
    if suggestions_df.empty:
        return pd.DataFrame(columns=["conference", "alias_name", "canonical_name", "score", "status"])

    top1 = suggestions_df[suggestions_df["rank"] == 1].copy()
    top2 = suggestions_df[suggestions_df["rank"] == 2][
        ["conference", "output_normalized_name", "score"]
    ].rename(columns={"score": "second_score"})

    top1 = top1.merge(top2, on=["conference", "output_normalized_name"], how="left")
    top1["second_score"] = top1["second_score"].fillna(0.0)

    strong_rule = (
        (top1["score"] >= PASS2_STRONG_THRESHOLD)
        & top1["last_name_exact"]
        & ((top1["score"] - top1["second_score"]) >= PASS2_MARGIN)
    )

    nickname_rule = (
        (top1["score"] >= PASS2_NICKNAME_THRESHOLD)
        & top1["last_name_exact"]
        & top1["first_name_equiv"]
        & ((top1["score"] - top1["second_score"]) >= PASS2_MARGIN)
    )

    accepted = top1[strong_rule | nickname_rule].copy()
    if accepted.empty:
        return pd.DataFrame(columns=["conference", "alias_name", "canonical_name", "score", "status"])

    accepted = accepted.sort_values(["conference", "score"], ascending=[True, False])
    accepted = accepted.drop_duplicates(subset=["conference", "output_normalized_name"], keep="first")
    accepted = accepted.rename(
        columns={
            "output_normalized_name": "alias_name",
            "truth_normalized_name": "canonical_name",
        }
    )[["conference", "alias_name", "canonical_name", "score"]]
    accepted["status"] = "conference_pass2_auto"
    return accepted.reset_index(drop=True)


# Base alias map from previous final review run, with fallback to file.
if "alias_union_final_df" not in globals() or alias_union_final_df.empty:
    alias_union_final_path = BASE / "participant_alias_mapping_final_review.csv"
    if alias_union_final_path.exists():
        alias_union_final_df = pd.read_csv(alias_union_final_path)
    else:
        alias_union_final_df = pd.DataFrame(columns=["conference", "alias_name", "canonical_name", "status"])

unmatched_base_df = unmatched_output_final_review_df.copy() if "unmatched_output_final_review_df" in globals() else pd.read_csv(BASE / "unmatched_output_participants_final_review.csv")

pass2_suggestions_df = build_pass2_suggestions(unmatched_base_df, truth_df, top_k=3)
pass2_accepted_df = accept_pass2_aliases(pass2_suggestions_df)

alias_union_pass2_df = pd.concat(
    [alias_union_final_df[["conference", "alias_name", "canonical_name", "status"]],
     pass2_accepted_df[["conference", "alias_name", "canonical_name", "status"]]],
    ignore_index=True,
)
alias_union_pass2_df = alias_union_pass2_df.drop_duplicates(
    subset=["conference", "alias_name"], keep="last"
).reset_index(drop=True)

output_df_pass2 = apply_auto_aliases(output_df, alias_union_pass2_df)

matched_pass2_df = output_df_pass2.merge(
    truth_df,
    on=["conference", "normalized_name"],
    how="inner",
)

unmatched_output_pass2_df = output_df_pass2.merge(
    truth_df[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_output_pass2_df = unmatched_output_pass2_df[
    unmatched_output_pass2_df["_merge"] == "left_only"
].drop(columns=["_merge"])

unmatched_truth_pass2_df = truth_df.merge(
    output_df_pass2[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_truth_pass2_df = unmatched_truth_pass2_df[
    unmatched_truth_pass2_df["_merge"] == "left_only"
].drop(columns=["_merge"])

progress_rows_pass2 = []
for conference in sorted(set(output_df_pass2["conference"]) | set(truth_df["conference"])):
    out_c = output_df_pass2[output_df_pass2["conference"] == conference]
    truth_c = truth_df[truth_df["conference"] == conference]
    matched_c = matched_pass2_df[matched_pass2_df["conference"] == conference]

    out_n = len(out_c)
    truth_n = len(truth_c)
    matched_n = len(matched_c)

    progress_rows_pass2.append(
        {
            "conference": conference,
            "output_unique_participants": out_n,
            "truth_unique_participants": truth_n,
            "matched_participants": matched_n,
            "unmatched_output_participants": out_n - matched_n,
            "unmatched_truth_participants": truth_n - matched_n,
            "output_match_rate": round(matched_n / out_n, 4) if out_n else 0.0,
            "truth_match_rate": round(matched_n / truth_n, 4) if truth_n else 0.0,
        }
    )

progress_pass2_df = pd.DataFrame(progress_rows_pass2).sort_values("conference").reset_index(drop=True)

pass2_suggestions_path = BASE / "conference_pass2_alias_suggestions.csv"
pass2_accepted_path = BASE / "conference_pass2_alias_accepted.csv"
matched_pass2_path = BASE / "matched_output_participants_conference_resolved.csv"
unmatched_output_pass2_path = BASE / "unmatched_output_participants_conference_resolved.csv"
unmatched_truth_pass2_path = BASE / "unmatched_truth_participants_conference_resolved.csv"
progress_pass2_path = BASE / "participant_matching_progress_by_conference_conference_resolved.csv"
alias_union_pass2_path = BASE / "participant_alias_mapping_conference_resolved.csv"

pass2_suggestions_df.to_csv(pass2_suggestions_path, index=False)
pass2_accepted_df.to_csv(pass2_accepted_path, index=False)
matched_pass2_df.to_csv(matched_pass2_path, index=False)
unmatched_output_pass2_df.to_csv(unmatched_output_pass2_path, index=False)
unmatched_truth_pass2_df.to_csv(unmatched_truth_pass2_path, index=False)
progress_pass2_df.to_csv(progress_pass2_path, index=False)
alias_union_pass2_df.to_csv(alias_union_pass2_path, index=False)

print(f"Saved: {pass2_suggestions_path}")
print(f"Saved: {pass2_accepted_path}")
print(f"Saved: {matched_pass2_path}")
print(f"Saved: {unmatched_output_pass2_path}")
print(f"Saved: {unmatched_truth_pass2_path}")
print(f"Saved: {progress_pass2_path}")
print(f"Saved: {alias_union_pass2_path}")
print(f"\nPass2 auto-accepted aliases: {len(pass2_accepted_df)}")
print(f"Unmatched output after pass2: {len(unmatched_output_pass2_df)}")
print(f"Unmatched truth after pass2: {len(unmatched_truth_pass2_df)}")

display(progress_pass2_df)

display(
    unmatched_output_pass2_df.groupby("conference", as_index=False)
    .size()
    .rename(columns={"size": "remaining_unmatched_output"})
    .sort_values("conference")
)

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/conference_pass2_alias_suggestions.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/conference_pass2_alias_accepted.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/matched_output_participants_conference_resolved.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_output_participants_conference_resolved.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_truth_participants_conference_resolved.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_matching_progress_by_conference_conference_resolved.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_alias_mapping_conference_resolved.csv

Pass2 auto-accepted aliases: 49
Unmatched output after pass2: 25
Unmatched truth after pass2: 0


,conference,output_unique_participants,truth_unique_participants,matched_participants,unmatched_output_participants,unmatched_truth_participants,output_match_rate,truth_match_rate
0,2020NES,101,94,94,7,0,0.9307,1.0
1,2021ABI,125,117,117,8,0,0.9360,1.0
2,2021CMC,77,77,77,0,0,1.0000,1.0
3,2021MND,103,98,98,5,0,0.9515,1.0
4,2021MZT,76,73,73,3,0,0.9605,1.0
5,2021NES,93,92,92,1,0,0.9892,1.0
6,2021SLU,81,81,81,0,0,1.0000,1.0
7,2022MND,55,54,54,1,0,0.9818,1.0


,conference,remaining_unmatched_output
0,2020NES,7
1,2021ABI,8
2,2021MND,5
3,2021MZT,3
4,2021NES,1
5,2022MND,1


## Review sheet for final manual reconciliation

This creates a single CSV to finish the remaining conference-level mismatches.

How to use:
1. Review `recommended_action` and top candidate columns.
2. Fill `decision` with one of: `accept_match`, `add_to_truth`, `exclude_generic`, `skip`.
3. If needed, set `final_canonical_name`.
4. Keep notes in `review_notes`.

Output:
- `manual_reconciliation_queue.csv`

In [11]:
# Build a review-ready queue from unresolved outputs after conference pass2.

if "unmatched_output_pass2_df" not in globals():
    unmatched_output_pass2_df = pd.read_csv(BASE / "unmatched_output_participants_conference_resolved.csv")
if "pass2_suggestions_df" not in globals():
    pass2_suggestions_df = pd.read_csv(BASE / "conference_pass2_alias_suggestions.csv")


def _recommend_action(name: str, top_score: float, second_score: float) -> str:
    if _is_placeholder_name(name):
        return "exclude_generic"
    if top_score >= 0.92 and (top_score - second_score) >= 0.07:
        return "accept_match"
    if top_score >= 0.86 and (top_score - second_score) >= 0.08:
        return "accept_match"
    if top_score <= 0.70:
        return "add_to_truth"
    return "review"


rank1 = pass2_suggestions_df[pass2_suggestions_df["rank"] == 1][
    ["conference", "output_normalized_name", "truth_normalized_name", "score", "truth_raw_names"]
].rename(
    columns={
        "truth_normalized_name": "top_truth_candidate",
        "score": "top_score",
        "truth_raw_names": "top_truth_raw_names",
    }
)

rank2 = pass2_suggestions_df[pass2_suggestions_df["rank"] == 2][
    ["conference", "output_normalized_name", "truth_normalized_name", "score"]
].rename(
    columns={
        "truth_normalized_name": "second_truth_candidate",
        "score": "second_score",
    }
)

review_queue_df = unmatched_output_pass2_df[
    [
        "conference",
        "normalized_name",
        "output_raw_names",
        "output_occurrence_count",
        "output_unique_file_count",
        "output_source_files",
    ]
].rename(columns={"normalized_name": "output_normalized_name"})

review_queue_df = review_queue_df.merge(
    rank1,
    on=["conference", "output_normalized_name"],
    how="left",
).merge(
    rank2,
    on=["conference", "output_normalized_name"],
    how="left",
)

review_queue_df["top_score"] = review_queue_df["top_score"].fillna(0.0)
review_queue_df["second_score"] = review_queue_df["second_score"].fillna(0.0)
review_queue_df["recommended_action"] = review_queue_df.apply(
    lambda row: _recommend_action(row["output_normalized_name"], row["top_score"], row["second_score"]),
    axis=1,
)

review_queue_df["decision"] = ""
review_queue_df["final_canonical_name"] = ""
review_queue_df["review_notes"] = ""

# Sort to prioritize highest-impact names first.
review_queue_df = review_queue_df.sort_values(
    ["conference", "output_occurrence_count", "top_score"],
    ascending=[True, False, False],
).reset_index(drop=True)

reconciliation_queue_path = BASE / "manual_reconciliation_queue.csv"
review_queue_df.to_csv(reconciliation_queue_path, index=False)
print(f"Saved: {reconciliation_queue_path}")
print(f"Rows to review: {len(review_queue_df)}")

print("\nRecommended action counts:")
print(review_queue_df["recommended_action"].value_counts(dropna=False).to_string())

print("\nRecommended action counts by conference:")
display(
    review_queue_df.groupby(["conference", "recommended_action"], as_index=False)
    .size()
    .rename(columns={"size": "rows"})
    .sort_values(["conference", "rows"], ascending=[True, False])
)

display(review_queue_df.head(30))

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/manual_reconciliation_queue.csv
Rows to review: 25

Recommended action counts:
recommended_action
accept_match       20
exclude_generic     5

Recommended action counts by conference:


,conference,recommended_action,rows
1,2020NES,exclude_generic,5
0,2020NES,accept_match,2
2,2021ABI,accept_match,8
3,2021MND,accept_match,5
4,2021MZT,accept_match,3
5,2021NES,accept_match,1
6,2022MND,accept_match,1


,conference,output_normalized_name,output_raw_names,output_occurrence_count,output_unique_file_count,output_source_files,top_truth_candidate,top_score,top_truth_raw_names,second_truth_candidate,second_score,recommended_action,decision,final_canonical_name,review_notes
0,2020NES,simona niguari,Simona Niguari,18,8,outputs/2020NES/output_2020_11_06_NES_S9/2020_...,simona liguori,0.8786,Simona Liguori,NaN,0.0000,accept_match,,,
1,2020NES,unknown,Unknown Speaker,10,8,outputs/2020NES/output_2020_11_06_NES_S9/2020_...,NaN,0.0000,NaN,NaN,0.0000,exclude_generic,,,
2,2020NES,david gator kwab,David Gator Kwab,9,3,outputs/2020NES/output_2020_11_06_NES_S12/6_ne...,david kwabi,0.9426,David Kwabi,david koweek,0.6643,accept_match,,,
3,2020NES,speaker 1,Speaker 1,9,3,outputs/2020NES/output_2020_11_05_NES_S4/4_Bey...,NaN,0.0000,NaN,NaN,0.0000,exclude_generic,,,
4,2020NES,speaker 2,Speaker 2,3,1,outputs/2020NES/output_2020_11_06_NES_S7/Zoom_...,NaN,0.0000,NaN,NaN,0.0000,exclude_generic,,,
5,2020NES,unknown female,Unknown Female,3,1,outputs/2020NES/output_2020_11_05_NES_S5/Zoom_...,NaN,0.0000,NaN,NaN,0.0000,exclude_generic,,,
6,2020NES,unknown male,Unknown Male,3,1,outputs/2020NES/output_2020_11_05_NES_S5/Zoom_...,NaN,0.0000,NaN,NaN,0.0000,exclude_generic,,,
7,2021ABI,mark seilmeyer,Mark Seilmeyer,33,8,outputs/2021ABI/output_2021_05_21_ABI_S8/botB3...,mark sellmyer,0.9395,Mark Sellmyer,NaN,0.0000,accept_match,,,
8,2021ABI,yevgenia kozoravitskiy,Yevgenia Kozoravitskiy,29,8,outputs/2021ABI/output_2021_05_21_ABI_S11/bot3...,yevgenia kozorovitskiy,1.0000,Yevgenia Kozorovitskiy,NaN,0.0000,accept_match,,,
9,2021ABI,shannon quine,Shannon Quine,27,9,outputs/2021ABI/output_2021_05_21_ABI_S1/Bot_4...,shannon quinn,0.9346,Shannon Quinn,NaN,0.0000,accept_match,,,


## Apply reconciliation decisions and finalize outputs

Run this after filling `manual_reconciliation_queue.csv`.

This cell:
- applies `accept_match` rows as new aliases,
- separates `exclude_generic` rows,
- exports `add_to_truth` candidates for truth dataset updates,
- writes finalized matched/unmatched outputs and conference progress.

In [12]:
queue_path = BASE / "manual_reconciliation_queue.csv"
if not queue_path.exists():
    raise FileNotFoundError(f"Missing reconciliation queue: {queue_path}")

if "output_df" not in globals() or "truth_df" not in globals():
    raise RuntimeError("Run the earlier data-loading cells first so output_df and truth_df are available.")

if "alias_union_pass2_df" not in globals() or alias_union_pass2_df.empty:
    alias_union_pass2_path = BASE / "participant_alias_mapping_conference_resolved.csv"
    if alias_union_pass2_path.exists():
        alias_union_pass2_df = pd.read_csv(alias_union_pass2_path)
    else:
        alias_union_pass2_df = pd.DataFrame(columns=["conference", "alias_name", "canonical_name", "status"])

queue_df = pd.read_csv(queue_path)
for column in ["decision", "final_canonical_name", "top_truth_candidate", "output_normalized_name"]:
    if column not in queue_df.columns:
        queue_df[column] = ""

queue_df["decision"] = queue_df["decision"].fillna("").astype(str).str.strip().str.lower()
queue_df["final_canonical_name"] = queue_df["final_canonical_name"].fillna("").astype(str).str.strip()
queue_df["top_truth_candidate"] = queue_df["top_truth_candidate"].fillna("").astype(str).str.strip()
queue_df["output_normalized_name"] = queue_df["output_normalized_name"].fillna("").astype(str).str.strip()

accepted_df = queue_df[queue_df["decision"] == "accept_match"].copy()
accepted_df["canonical_name"] = accepted_df["final_canonical_name"].where(
    accepted_df["final_canonical_name"] != "",
    accepted_df["top_truth_candidate"],
)
accepted_df = accepted_df[accepted_df["canonical_name"] != ""]

manual_accept_alias_df = accepted_df[["conference", "output_normalized_name", "canonical_name"]].rename(
    columns={"output_normalized_name": "alias_name"}
)
manual_accept_alias_df["status"] = "manual_reconciliation_accept"

alias_final_df = pd.concat(
    [
        alias_union_pass2_df[["conference", "alias_name", "canonical_name", "status"]],
        manual_accept_alias_df[["conference", "alias_name", "canonical_name", "status"]],
    ],
    ignore_index=True,
)
alias_final_df = alias_final_df.drop_duplicates(subset=["conference", "alias_name"], keep="last").reset_index(drop=True)

output_df_finalized = apply_auto_aliases(output_df, alias_final_df)

matched_finalized_df = output_df_finalized.merge(
    truth_df,
    on=["conference", "normalized_name"],
    how="inner",
)

unmatched_output_finalized_df = output_df_finalized.merge(
    truth_df[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_output_finalized_df = unmatched_output_finalized_df[
    unmatched_output_finalized_df["_merge"] == "left_only"
].drop(columns=["_merge"])

unmatched_truth_finalized_df = truth_df.merge(
    output_df_finalized[["conference", "normalized_name"]],
    on=["conference", "normalized_name"],
    how="left",
    indicator=True,
)
unmatched_truth_finalized_df = unmatched_truth_finalized_df[
    unmatched_truth_finalized_df["_merge"] == "left_only"
].drop(columns=["_merge"])

exclude_generic_df = queue_df[queue_df["decision"] == "exclude_generic"].copy()
exclude_generic_df = exclude_generic_df[["conference", "output_normalized_name", "output_raw_names", "review_notes"]].drop_duplicates()

if not exclude_generic_df.empty:
    unmatched_output_finalized_filtered_df = unmatched_output_finalized_df.merge(
        exclude_generic_df[["conference", "output_normalized_name"]].rename(columns={"output_normalized_name": "normalized_name"}),
        on=["conference", "normalized_name"],
        how="left",
        indicator=True,
    )
    unmatched_output_finalized_filtered_df = unmatched_output_finalized_filtered_df[
        unmatched_output_finalized_filtered_df["_merge"] == "left_only"
    ].drop(columns=["_merge"])
else:
    unmatched_output_finalized_filtered_df = unmatched_output_finalized_df.copy()

add_to_truth_df = queue_df[queue_df["decision"] == "add_to_truth"].copy()
add_to_truth_df = add_to_truth_df[
    [
        "conference",
        "output_normalized_name",
        "output_raw_names",
        "output_occurrence_count",
        "output_unique_file_count",
        "output_source_files",
        "review_notes",
    ]
].rename(columns={"output_normalized_name": "candidate_truth_name"})

pending_review_df = queue_df[queue_df["decision"].isin(["", "review", "skip"])].copy()

progress_rows_finalized = []
for conference in sorted(set(output_df_finalized["conference"]) | set(truth_df["conference"])):
    out_c = output_df_finalized[output_df_finalized["conference"] == conference]
    truth_c = truth_df[truth_df["conference"] == conference]
    matched_c = matched_finalized_df[matched_finalized_df["conference"] == conference]

    out_n = len(out_c)
    truth_n = len(truth_c)
    matched_n = len(matched_c)

    progress_rows_finalized.append(
        {
            "conference": conference,
            "output_unique_participants": out_n,
            "truth_unique_participants": truth_n,
            "matched_participants": matched_n,
            "unmatched_output_participants": out_n - matched_n,
            "unmatched_truth_participants": truth_n - matched_n,
            "output_match_rate": round(matched_n / out_n, 4) if out_n else 0.0,
            "truth_match_rate": round(matched_n / truth_n, 4) if truth_n else 0.0,
        }
    )

progress_finalized_df = pd.DataFrame(progress_rows_finalized).sort_values("conference").reset_index(drop=True)

alias_final_path = BASE / "participant_alias_mapping_finalized.csv"
matched_finalized_path = BASE / "matched_output_participants_finalized.csv"
unmatched_output_finalized_path = BASE / "unmatched_output_participants_finalized.csv"
unmatched_output_filtered_path = BASE / "unmatched_output_participants_finalized_filtered.csv"
unmatched_truth_finalized_path = BASE / "unmatched_truth_participants_finalized.csv"
progress_finalized_path = BASE / "participant_matching_progress_by_conference_finalized.csv"
add_to_truth_path = BASE / "truth_addition_candidates.csv"
exclude_generic_path = BASE / "excluded_generic_output_names.csv"
pending_review_path = BASE / "manual_reconciliation_pending.csv"

alias_final_df.to_csv(alias_final_path, index=False)
matched_finalized_df.to_csv(matched_finalized_path, index=False)
unmatched_output_finalized_df.to_csv(unmatched_output_finalized_path, index=False)
unmatched_output_finalized_filtered_df.to_csv(unmatched_output_filtered_path, index=False)
unmatched_truth_finalized_df.to_csv(unmatched_truth_finalized_path, index=False)
progress_finalized_df.to_csv(progress_finalized_path, index=False)
add_to_truth_df.to_csv(add_to_truth_path, index=False)
exclude_generic_df.to_csv(exclude_generic_path, index=False)
pending_review_df.to_csv(pending_review_path, index=False)

print(f"Saved: {alias_final_path}")
print(f"Saved: {matched_finalized_path}")
print(f"Saved: {unmatched_output_finalized_path}")
print(f"Saved: {unmatched_output_filtered_path}")
print(f"Saved: {unmatched_truth_finalized_path}")
print(f"Saved: {progress_finalized_path}")
print(f"Saved: {add_to_truth_path}")
print(f"Saved: {exclude_generic_path}")
print(f"Saved: {pending_review_path}")

print("\nDecision counts:")
print(queue_df["decision"].replace({"": "blank"}).value_counts(dropna=False).to_string())

print("\nFinalized summary:")
print(f"- matched participants: {len(matched_finalized_df)}")
print(f"- unmatched output (raw): {len(unmatched_output_finalized_df)}")
print(f"- unmatched output (excluding generic): {len(unmatched_output_finalized_filtered_df)}")
print(f"- unmatched truth: {len(unmatched_truth_finalized_df)}")

display(progress_finalized_df)


Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_alias_mapping_finalized.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/matched_output_participants_finalized.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_output_participants_finalized.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_output_participants_finalized_filtered.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/unmatched_truth_participants_finalized.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/participant_matching_progress_by_conference_finalized.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/truth_addition_candidates.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/excluded_generic_output_names.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/manual_reconciliation_pending.csv

Decision counts:
decision
blank    25

Finalized summary:
- matched participants: 686
- unmatched output (ra

,conference,output_unique_participants,truth_unique_participants,matched_participants,unmatched_output_participants,unmatched_truth_participants,output_match_rate,truth_match_rate
0,2020NES,101,94,94,7,0,0.9307,1.0
1,2021ABI,125,117,117,8,0,0.9360,1.0
2,2021CMC,77,77,77,0,0,1.0000,1.0
3,2021MND,103,98,98,5,0,0.9515,1.0
4,2021MZT,76,73,73,3,0,0.9605,1.0
5,2021NES,93,92,92,1,0,0.9892,1.0
6,2021SLU,81,81,81,0,0,1.0000,1.0
7,2022MND,55,54,54,1,0,0.9818,1.0


## Conservative typo sweep on finalized matched output

This optional post-processing pass catches typo-level duplicates conservatively.

Rules:
1. Compare names only within the same conference.
2. Require same token count and identical non-surname tokens.
3. Require high surname similarity.
4. Keep short/ambiguous names out by requiring at least two tokens.

Outputs:
- `conservative_typo_sweep_candidates.csv`
- `conservative_typo_sweep_application_report.csv`
- Backup of `matched_output_participants_finalized.csv` before edits

In [13]:
from difflib import SequenceMatcher
from datetime import datetime

# Prefer finalized_matching_csvs if present; otherwise fall back to BASE root outputs.
FINALIZED_DIR = BASE / "finalized_matching_csvs"
FINALIZED_DIR = FINALIZED_DIR if FINALIZED_DIR.exists() else BASE

matched_finalized_path = FINALIZED_DIR / "matched_output_participants_finalized.csv"
if not matched_finalized_path.exists():
    raise FileNotFoundError(f"Missing finalized matched file: {matched_finalized_path}")

SURNAME_SIMILARITY_THRESHOLD = 0.86

finalized_df = pd.read_csv(matched_finalized_path)

candidate_rows = []
for conference, part in finalized_df.groupby("conference"):
    rows = part[
        [
            "normalized_name",
            "output_occurrence_count",
            "truth_occurrence_count",
        ]
    ].fillna(0).to_dict("records")

    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            name_a = str(rows[i]["normalized_name"]).strip()
            name_b = str(rows[j]["normalized_name"]).strip()

            tokens_a = name_a.split()
            tokens_b = name_b.split()

            if len(tokens_a) < 2 or len(tokens_b) < 2:
                continue
            if len(tokens_a) != len(tokens_b):
                continue
            if tokens_a[:-1] != tokens_b[:-1]:
                continue
            if tokens_a[-1] == tokens_b[-1]:
                continue

            surname_similarity = SequenceMatcher(None, tokens_a[-1], tokens_b[-1]).ratio()
            if surname_similarity < SURNAME_SIMILARITY_THRESHOLD:
                continue

            total_signal_a = float(rows[i].get("output_occurrence_count", 0) or 0) + float(
                rows[i].get("truth_occurrence_count", 0) or 0
            )
            total_signal_b = float(rows[j].get("output_occurrence_count", 0) or 0) + float(
                rows[j].get("truth_occurrence_count", 0) or 0
            )

            canonical_candidate = name_a if total_signal_a >= total_signal_b else name_b
            alias_candidate = name_b if canonical_candidate == name_a else name_a

            candidate_rows.append(
                {
                    "conference": conference,
                    "name_a": name_a,
                    "name_b": name_b,
                    "surname_similarity": round(float(surname_similarity), 4),
                    "canonical_candidate": canonical_candidate,
                    "alias_candidate": alias_candidate,
                    "total_signal_a": total_signal_a,
                    "total_signal_b": total_signal_b,
                }
            )

candidates_df = (
    pd.DataFrame(candidate_rows)
    .sort_values(["conference", "surname_similarity"], ascending=[True, False])
    .reset_index(drop=True)
    if candidate_rows
    else pd.DataFrame(
        columns=[
            "conference",
            "name_a",
            "name_b",
            "surname_similarity",
            "canonical_candidate",
            "alias_candidate",
            "total_signal_a",
            "total_signal_b",
        ]
    )
)

candidates_path = FINALIZED_DIR / "conservative_typo_sweep_candidates.csv"
candidates_df.to_csv(candidates_path, index=False)

# Apply exactly the candidates generated in this pass.
apply_rows = []
updated_df = finalized_df.copy()
for _, row in candidates_df.iterrows():
    conference = str(row["conference"])
    alias_name = str(row["alias_candidate"]).strip()
    canonical_name = str(row["canonical_candidate"]).strip()

    alias_mask = (
        (updated_df["conference"].astype(str) == conference)
        & (updated_df["normalized_name"].astype(str).str.lower() == alias_name.lower())
    )
    alias_rows_found = int(alias_mask.sum())
    canonical_rows_preexisting = int(
        (
            (updated_df["conference"].astype(str) == conference)
            & (
                updated_df["normalized_name"].astype(str).str.lower()
                == canonical_name.lower()
            )
        ).sum()
    )

    status = "applied" if alias_rows_found > 0 else "skipped_alias_not_found"
    if alias_rows_found > 0:
        updated_df.loc[alias_mask, "normalized_name"] = canonical_name

    apply_rows.append(
        {
            "conference": conference,
            "alias_name": alias_name,
            "canonical_name": canonical_name,
            "alias_rows_found": alias_rows_found,
            "canonical_rows_preexisting": canonical_rows_preexisting,
            "status": status,
            "timestamp_utc": datetime.utcnow().isoformat(),
        }
    )

backup_path = FINALIZED_DIR / "matched_output_participants_finalized.pre_conservative_typo_sweep.csv"
if not backup_path.exists():
    finalized_df.to_csv(backup_path, index=False)

agg_spec = {
    "output_raw_names": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "output_sources": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "output_source_files": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "output_occurrence_count": "sum",
    "output_unique_file_count": "sum",
    "truth_raw_names": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "truth_sources": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "truth_source_files": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "truth_occurrence_count": "sum",
    "truth_unique_file_count": "sum",
}
agg_spec = {key: value for key, value in agg_spec.items() if key in updated_df.columns}

finalized_updated_df = (
    updated_df.groupby(["conference", "normalized_name"], as_index=False)
    .agg(agg_spec)
    .sort_values(["conference", "normalized_name"])
    .reset_index(drop=True)
)
finalized_updated_df.to_csv(matched_finalized_path, index=False)

apply_report_df = pd.DataFrame(apply_rows)
apply_report_path = FINALIZED_DIR / "conservative_typo_sweep_application_report.csv"
apply_report_df.to_csv(apply_report_path, index=False)

print(f"Saved: {candidates_path}")
print(f"Saved: {apply_report_path}")
print(f"Saved: {matched_finalized_path}")
print(f"Backup: {backup_path}")
print(f"Candidates found: {len(candidates_df)}")
print(f"Rows before: {len(finalized_df)}")
print(f"Rows after: {len(finalized_updated_df)}")

if not apply_report_df.empty:
    print("\nApplication status counts:")
    print(apply_report_df["status"].value_counts(dropna=False).to_string())

display(candidates_df.head(30))

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/conservative_typo_sweep_candidates.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/conservative_typo_sweep_application_report.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/matched_output_participants_finalized.csv
Backup: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/matched_output_participants_finalized.pre_conservative_typo_sweep.csv
Candidates found: 0
Rows before: 682
Rows after: 682


,conference,name_a,name_b,surname_similarity,canonical_candidate,alias_candidate,total_signal_a,total_signal_b


## Cross-conference global person identity view

This section creates a global participant table where the same normalized person is counted once across all conferences.

It does not replace conference-level outputs; it adds a deduplicated global view.

Outputs:
- `global_participant_identity_map.csv`
- `global_participant_identity_by_conference.csv`

In [14]:
# Build a cross-conference person identity map (one row per normalized person).

finalized_dir = BASE / "finalized_matching_csvs"
finalized_dir = finalized_dir if finalized_dir.exists() else BASE

matched_root_path = BASE / "matched_output_participants_finalized.csv"
curated_matched_path = finalized_dir / "matched_output_participants_finalized.csv"

if matched_root_path.exists():
    global_source_df = pd.read_csv(matched_root_path)
elif curated_matched_path.exists():
    global_source_df = pd.read_csv(curated_matched_path)
elif "matched_finalized_df" in globals() and not matched_finalized_df.empty:
    global_source_df = matched_finalized_df.copy()
else:
    raise FileNotFoundError(
        "No finalized matched participant data found in root/finalized paths and matched_finalized_df is unavailable."
    )

required_cols = {"conference", "normalized_name"}
missing = [c for c in required_cols if c not in global_source_df.columns]
if missing:
    raise ValueError(f"Global identity mapping requires columns: {missing}")

global_source_df = global_source_df.copy()
global_source_df["conference"] = global_source_df["conference"].astype(str).str.strip()
global_source_df["normalized_name"] = global_source_df["normalized_name"].astype(str).str.strip().str.lower()

for numeric_col in [
    "output_occurrence_count",
    "truth_occurrence_count",
    "output_unique_file_count",
    "truth_unique_file_count",
]:
    if numeric_col in global_source_df.columns:
        global_source_df[numeric_col] = pd.to_numeric(global_source_df[numeric_col], errors="coerce").fillna(0)


def _merge_pipe_values(values):
    items = set()
    for value in values:
        if pd.isna(value):
            continue
        for part in str(value).split(" | "):
            part = part.strip()
            if part:
                items.add(part)
    return " | ".join(sorted(items))


agg_spec = {
    "conferences": ("conference", lambda v: " | ".join(sorted(set(v)))),
    "conference_count": ("conference", lambda v: len(set(v))),
}

if "output_raw_names" in global_source_df.columns:
    agg_spec["output_raw_names_all"] = ("output_raw_names", _merge_pipe_values)
if "truth_raw_names" in global_source_df.columns:
    agg_spec["truth_raw_names_all"] = ("truth_raw_names", _merge_pipe_values)
if "output_source_files" in global_source_df.columns:
    agg_spec["output_source_files_all"] = ("output_source_files", _merge_pipe_values)
if "truth_source_files" in global_source_df.columns:
    agg_spec["truth_source_files_all"] = ("truth_source_files", _merge_pipe_values)
if "output_occurrence_count" in global_source_df.columns:
    agg_spec["output_occurrence_count_total"] = ("output_occurrence_count", "sum")
if "truth_occurrence_count" in global_source_df.columns:
    agg_spec["truth_occurrence_count_total"] = ("truth_occurrence_count", "sum")
if "output_unique_file_count" in global_source_df.columns:
    agg_spec["output_unique_file_count_total"] = ("output_unique_file_count", "sum")
if "truth_unique_file_count" in global_source_df.columns:
    agg_spec["truth_unique_file_count_total"] = ("truth_unique_file_count", "sum")

global_identity_df = (
    global_source_df.groupby("normalized_name", as_index=False)
    .agg(**agg_spec)
    .sort_values(["conference_count", "normalized_name"], ascending=[False, True])
    .reset_index(drop=True)
)

global_identity_df.insert(
    0,
    "global_person_id",
    [f"person_{i:04d}" for i in range(1, len(global_identity_df) + 1)],
)

# Expanded mapping table: one row per person per conference.
global_identity_by_conference_df = global_source_df[["normalized_name", "conference"]].drop_duplicates()
global_identity_by_conference_df = global_identity_by_conference_df.merge(
    global_identity_df[["global_person_id", "normalized_name"]],
    on="normalized_name",
    how="left",
)
global_identity_by_conference_df = global_identity_by_conference_df[
    ["global_person_id", "normalized_name", "conference"]
].sort_values(["global_person_id", "conference"]).reset_index(drop=True)

identity_map_path = finalized_dir / "global_participant_identity_map.csv"
identity_by_conference_path = finalized_dir / "global_participant_identity_by_conference.csv"

global_identity_df.to_csv(identity_map_path, index=False)
global_identity_by_conference_df.to_csv(identity_by_conference_path, index=False)

print(f"Saved: {identity_map_path}")
print(f"Saved: {identity_by_conference_path}")
print(f"Global unique people: {len(global_identity_df)}")
print(f"Conference-level rows (person x conference): {len(global_identity_by_conference_df)}")

display(global_identity_df.head(20))

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/global_participant_identity_map.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/global_participant_identity_by_conference.csv
Global unique people: 550
Conference-level rows (person x conference): 686


,global_person_id,normalized_name,conferences,conference_count,output_raw_names_all,truth_raw_names_all,output_source_files_all,truth_source_files_all,output_occurrence_count_total,truth_occurrence_count_total,output_unique_file_count_total,truth_unique_file_count_total
0,person_0001,andrew feig,2020NES | 2021ABI | 2021CMC | 2021MND | 2021MZ...,8,Andrew Feig,andrew feig,outputs/2020NES/output_2020_11_05_NES_S5/5_Dec...,analysis_v1/data/2020NES/2020NES_outcome.json ...,455,8,189,8
1,person_0002,richard wiener,2020NES | 2021ABI | 2021CMC | 2021MND | 2021MZ...,8,Richard Wiener,richard wiener,outputs/2020NES/output_2020_11_06_NES_S2/2020_...,analysis_v1/data/2020NES/2020NES_outcome.json ...,479,8,195,8
2,person_0003,silvia ronco,2020NES | 2021ABI | 2021CMC | 2021MND | 2021MZ...,8,Silvia Ronco,silvia ronco,outputs/2020NES/output_2020_11_06_NES_S2/2020_...,analysis_v1/data/2020NES/2020NES_outcome.json ...,200,8,70,8
3,person_0004,sandra laney,2021ABI | 2021CMC | 2021MND | 2021MZT | 2022MND,5,Sandra Laney,sandra laney,outputs/2021ABI/output_2021_05_20_ABI_S4/bot2p...,analysis_v1/data/2021ABI/2021ABI_outcome.json ...,114,5,80,5
4,person_0005,alexandra basford,2021ABI | 2021MND | 2022MND,3,Alexandra Basford,alexandra basford,outputs/2021ABI/output_2021_05_21_ABI_S5/bot5p...,analysis_v1/data/2021ABI/2021ABI_outcome.json ...,37,3,29,3
5,person_0006,ali keshavarzian,2021ABI | 2021MND | 2022MND,3,Ali Keshavarzian,ali keshavarzian,outputs/2021ABI/output_2021_05_21_ABI_S16/Brea...,analysis_v1/data/2021ABI/2021ABI_outcome.json ...,300,3,48,3
6,person_0007,carolina tropini,2021ABI | 2021MND | 2022MND,3,Carolina Tropini,Carolina Tropini | carolina tropini,outputs/2021ABI/output_2021_05_21_ABI_S16/Brea...,analysis_v1/data/2021ABI/2021ABI_outcome.json ...,135,3,36,3
7,person_0008,daren ginete,2021CMC | 2021MND | 2021MZT,3,Daren Ginete,daren ginete,outputs/2021CMC/output_2021_10_08_CMC_S3/B2_Zo...,analysis_v1/data/2021CMC/2021CMC_outcome.json ...,49,3,35,3
8,person_0009,lisa ryno,2021ABI | 2021MND | 2022MND,3,Lisa Ryno,lisa ryno,outputs/2021ABI/output_2021_05_21_ABI_S16/Brea...,analysis_v1/data/2021ABI/2021ABI_outcome.json ...,115,3,41,3
9,person_0010,stavroula hatzios,2021ABI | 2021MND | 2022MND,3,Stavroula Hatzios,Stavroula Hatzios | stavroula hatzios,outputs/2021ABI/output_2021_05_21_ABI_S16/Brea...,analysis_v1/data/2021ABI/2021ABI_outcome.json ...,140,3,50,3


## Additional manual name-group pass (latest review)

This pass applies your latest name-group hints as *equivalence groups*.

Behavior:
1. Treat names in each row as likely same person.
2. Choose a canonical name per group based on names already present in the finalized matched CSV.
3. Apply only to the curated finalized matched file and write new output files (does not overwrite baseline finalized files).

Outputs:
- `matched_output_participants_finalized_user_review_pass.csv`
- `user_review_name_group_application_report.csv`
- `global_participant_identity_map_user_review_pass.csv`
- `global_participant_identity_by_conference_user_review_pass.csv`

In [15]:
import re

finalized_dir = BASE / "finalized_matching_csvs"
finalized_dir = finalized_dir if finalized_dir.exists() else BASE

baseline_matched_path = finalized_dir / "matched_output_participants_finalized.csv"
if not baseline_matched_path.exists():
    raise FileNotFoundError(f"Missing curated finalized file: {baseline_matched_path}")

baseline_df = pd.read_csv(baseline_matched_path).copy()


def _norm_name(value: str) -> str:
    text = "" if pd.isna(value) else str(value)
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _merge_pipe_values(values):
    items = set()
    for value in values:
        if pd.isna(value):
            continue
        for part in str(value).split(" | "):
            part = part.strip()
            if part:
                items.add(part)
    return " | ".join(sorted(items))


# Each sublist is one user-provided equivalence group.
USER_NAME_GROUPS = [
    ["Arnold", "Arnold Hayer"],
    ["Barbara Smith", "Brad Smith"],
    ["Dave Durgan", "David Durgan"],
    ["Jeff Long", "Jeff Long - UC Berkeley", "Jeffrey Long", "Jeffrey Long UC Berkeley", "Jeffrey R Long"],
    ["Jae Sung", "Jaeyun Sun"],
    ["Julia", "Julia Brown"],
    ["Kelsey Hatzell", "Kelsey Hazell", "Kelsey B Hatzell"],
    ["Phil Milner", "Phillip Milner"],
    ["Sarah", "Sarah Maceachern"],
    ["Shiva Abbasi", "Shiva Abbaszadeh"],
    ["Troy Clavell Sutton", "Troy Sutton"],
    ["Ukpong Eyo", "Ukpong Euo-UofVirginia"],
    ["Gary Moore", "Gary F Moore"],
    ["Wilson Smith", "Wilson A Smith"],
]

baseline_df["normalized_name"] = baseline_df["normalized_name"].astype(str).str.strip().str.lower()
for numeric_col in ["output_occurrence_count", "truth_occurrence_count"]:
    if numeric_col in baseline_df.columns:
        baseline_df[numeric_col] = pd.to_numeric(baseline_df[numeric_col], errors="coerce").fillna(0)

# Compute total signal per normalized name to choose canonical names consistently.
name_signal_df = baseline_df.groupby("normalized_name", as_index=False).agg(
    output_signal=("output_occurrence_count", "sum") if "output_occurrence_count" in baseline_df.columns else ("normalized_name", "count"),
    truth_signal=("truth_occurrence_count", "sum") if "truth_occurrence_count" in baseline_df.columns else ("normalized_name", "count"),
)
name_signal_df["total_signal"] = name_signal_df["output_signal"] + name_signal_df["truth_signal"]
name_to_signal = dict(zip(name_signal_df["normalized_name"], name_signal_df["total_signal"]))

alias_to_canonical = {}
report_rows = []

for group_index, raw_group in enumerate(USER_NAME_GROUPS, start=1):
    group_norm = [_norm_name(name) for name in raw_group if _norm_name(name)]
    group_norm = list(dict.fromkeys(group_norm))
    present = [name for name in group_norm if name in name_to_signal]

    if present:
        canonical = sorted(
            present,
            key=lambda name: (
                -float(name_to_signal.get(name, 0)),
                -len(name.split()),
                name,
            ),
        )[0]
        status = "applied_group"
    else:
        canonical = group_norm[0] if group_norm else ""
        status = "skipped_no_names_present"

    for name in group_norm:
        if not name:
            continue
        if name == canonical:
            row_status = "canonical_choice"
        elif name in name_to_signal:
            alias_to_canonical[name] = canonical
            row_status = "alias_applied"
        else:
            row_status = "name_not_present"

        report_rows.append(
            {
                "group_id": group_index,
                "group_names": " | ".join(group_norm),
                "name": name,
                "canonical_name": canonical,
                "name_present_in_baseline": bool(name in name_to_signal),
                "name_total_signal": float(name_to_signal.get(name, 0)),
                "group_status": status,
                "row_status": row_status,
            }
        )

updated_df = baseline_df.copy()
updated_df["normalized_name"] = updated_df["normalized_name"].map(lambda name: alias_to_canonical.get(name, name))

agg_spec = {
    "output_raw_names": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "output_sources": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "output_source_files": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "output_occurrence_count": "sum",
    "output_unique_file_count": "sum",
    "truth_raw_names": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "truth_sources": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "truth_source_files": lambda v: " | ".join(sorted(set(" | ".join(map(str, v)).split(" | ")))),
    "truth_occurrence_count": "sum",
    "truth_unique_file_count": "sum",
}
agg_spec = {k: v for k, v in agg_spec.items() if k in updated_df.columns}

updated_df = (
    updated_df.groupby(["conference", "normalized_name"], as_index=False)
    .agg(agg_spec)
    .sort_values(["conference", "normalized_name"])
    .reset_index(drop=True)
)

user_pass_matched_path = finalized_dir / "matched_output_participants_finalized_user_review_pass.csv"
user_pass_report_path = finalized_dir / "user_review_name_group_application_report.csv"

updated_df.to_csv(user_pass_matched_path, index=False)
pd.DataFrame(report_rows).to_csv(user_pass_report_path, index=False)

# Also write global identity outputs based on this updated user-pass file.
identity_source_df = updated_df.copy()
identity_source_df["conference"] = identity_source_df["conference"].astype(str).str.strip()
identity_source_df["normalized_name"] = identity_source_df["normalized_name"].astype(str).str.strip().str.lower()

for numeric_col in [
    "output_occurrence_count",
    "truth_occurrence_count",
    "output_unique_file_count",
    "truth_unique_file_count",
]:
    if numeric_col in identity_source_df.columns:
        identity_source_df[numeric_col] = pd.to_numeric(identity_source_df[numeric_col], errors="coerce").fillna(0)

identity_agg = {
    "conferences": ("conference", lambda v: " | ".join(sorted(set(v)))),
    "conference_count": ("conference", lambda v: len(set(v))),
}
if "output_raw_names" in identity_source_df.columns:
    identity_agg["output_raw_names_all"] = ("output_raw_names", _merge_pipe_values)
if "truth_raw_names" in identity_source_df.columns:
    identity_agg["truth_raw_names_all"] = ("truth_raw_names", _merge_pipe_values)
if "output_source_files" in identity_source_df.columns:
    identity_agg["output_source_files_all"] = ("output_source_files", _merge_pipe_values)
if "truth_source_files" in identity_source_df.columns:
    identity_agg["truth_source_files_all"] = ("truth_source_files", _merge_pipe_values)
if "output_occurrence_count" in identity_source_df.columns:
    identity_agg["output_occurrence_count_total"] = ("output_occurrence_count", "sum")
if "truth_occurrence_count" in identity_source_df.columns:
    identity_agg["truth_occurrence_count_total"] = ("truth_occurrence_count", "sum")
if "output_unique_file_count" in identity_source_df.columns:
    identity_agg["output_unique_file_count_total"] = ("output_unique_file_count", "sum")
if "truth_unique_file_count" in identity_source_df.columns:
    identity_agg["truth_unique_file_count_total"] = ("truth_unique_file_count", "sum")

identity_map_df = (
    identity_source_df.groupby("normalized_name", as_index=False)
    .agg(**identity_agg)
    .sort_values(["conference_count", "normalized_name"], ascending=[False, True])
    .reset_index(drop=True)
)
identity_map_df.insert(0, "global_person_id", [f"person_{i:04d}" for i in range(1, len(identity_map_df) + 1)])

identity_by_conf_df = identity_source_df[["normalized_name", "conference"]].drop_duplicates()
identity_by_conf_df = identity_by_conf_df.merge(
    identity_map_df[["global_person_id", "normalized_name"]],
    on="normalized_name",
    how="left",
)
identity_by_conf_df = identity_by_conf_df[["global_person_id", "normalized_name", "conference"]].sort_values(
    ["global_person_id", "conference"]
).reset_index(drop=True)

identity_map_path = finalized_dir / "global_participant_identity_map_user_review_pass.csv"
identity_by_conf_path = finalized_dir / "global_participant_identity_by_conference_user_review_pass.csv"
identity_map_df.to_csv(identity_map_path, index=False)
identity_by_conf_df.to_csv(identity_by_conf_path, index=False)

print(f"Saved: {user_pass_matched_path}")
print(f"Saved: {user_pass_report_path}")
print(f"Saved: {identity_map_path}")
print(f"Saved: {identity_by_conf_path}")
print(f"Rows before user pass: {len(baseline_df)}")
print(f"Rows after user pass: {len(updated_df)}")
print(f"Global unique people after user pass: {len(identity_map_df)}")

report_df = pd.DataFrame(report_rows)
print("\nUser-group application row status counts:")
print(report_df["row_status"].value_counts(dropna=False).to_string())

display(report_df)

Saved: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/matched_output_participants_finalized_user_review_pass.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/user_review_name_group_application_report.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/global_participant_identity_map_user_review_pass.csv
Saved: /Users/maxchalekson/Projects/gemini_data_analysis/finalized_matching_csvs/global_participant_identity_by_conference_user_review_pass.csv
Rows before user pass: 682
Rows after user pass: 679
Global unique people after user pass: 543

User-group application row status counts:
row_status
canonical_choice    14
name_not_present    12
alias_applied        6


,group_id,group_names,name,canonical_name,name_present_in_baseline,name_total_signal,group_status,row_status
0,1,arnold | arnold hayer,arnold,arnold hayer,False,0.0,applied_group,name_not_present
1,1,arnold | arnold hayer,arnold hayer,arnold hayer,True,92.0,applied_group,canonical_choice
2,2,barbara smith | brad smith,barbara smith,barbara smith,True,330.0,applied_group,canonical_choice
3,2,barbara smith | brad smith,brad smith,barbara smith,False,0.0,applied_group,name_not_present
4,3,dave durgan | david durgan,dave durgan,david durgan,False,0.0,applied_group,name_not_present
5,3,dave durgan | david durgan,david durgan,david durgan,True,158.0,applied_group,canonical_choice
6,4,jeff long | jeff long uc berkeley | jeffrey lo...,jeff long,jeffrey long,True,91.0,applied_group,alias_applied
7,4,jeff long | jeff long uc berkeley | jeffrey lo...,jeff long uc berkeley,jeffrey long,False,0.0,applied_group,name_not_present
8,4,jeff long | jeff long uc berkeley | jeffrey lo...,jeffrey long,jeffrey long,True,162.0,applied_group,canonical_choice
9,4,jeff long | jeff long uc berkeley | jeffrey lo...,jeffrey long uc berkeley,jeffrey long,True,100.0,applied_group,alias_applied
